In [ ]:
# CELL 1 — FINAL LOCKED TEST AUTHORIZATION
import hashlib

AUTHORIZE_LOCKED_TEST = True
LOCKED_TEST_ACK = "I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST"

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
# Historical standalone-Gated audit references only. The final resolver does NOT
# use these tables to accept/reject a FacetGate package; each completed package
# cryptographically binds its own exact same-seed Gated parent.
CURRENT_FROZEN_PARENT_GATED_STEPS = {42: 932, 1337: 800, 2026: 600, 31415: 932, 271828: 932}
CURRENT_FROZEN_PARENT_FINGERPRINTS = {
    42: '8f3d833bf46b2906de3adb7dd25bcdee580335afcee165f75fc62ee71dacad22',
    1337: '2c5b889fda8cff3ed6132cb12cb8c9d5c0306d8572c87177f39b21e9d30026ca',
    2026: '67c11cc5b032d58a4361ff4828724fc8def57ec077861c36ad3d1eb647aede82',
    31415: 'fed7ead35a499bba074006077baab1bef4a52912ab28366883b2ec16c4a29b09',
    271828: 'd199410f6af61b8db9c67633084c1f43f126a6f8dfd3509392a6ea0a8790e347',
}

MODEL_ID = 'OpenGVLab/InternVL3_5-8B-HF'
MODEL_REVISION = '3974c115d44f0e13eae8b654f680b1e561c71ae9'
EXPECTED_HIDDEN = 4096

FACETGATE_COMMON_SHA256 = '735b3e08413949cdb148fe4ce570493b5cd410e562dad7a12748b2e5f2fd6a38'
PARENT_GATED_PROTOCOL_SHA256 = '627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd'
SHARED_FORWARD_PROTOCOL_SHA256 = '37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa'
TASK_INSTRUCTION_SHA256 = 'bfef1e522a9433e1ca0c57a3a1bea49cd890d8428308b7ffdf6b24e4fe51667f'

EXPECTED_LABEL_SHA256 = '9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6'
EXPECTED_OCR_SHA256 = 'eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c'
EXPECTED_SPLIT_SHA256 = 'bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0'
EXPECTED_CANONICAL_ROW_SHA256 = '440f8e6982cf4272fc0b6c3e3e6fc9b079a02ce01cd5350a02be0d4a9fa1cf39'
EXPECTED_TEST_IDENTITY_SHA256 = '5f6ac881e9026d682f2f41d65ed1f08e1861d8c8f030bae04d98f9b9db6419db'
EXPECTED_IMAGE_BUNDLE_SHA256 = '9e134f26c63c23ba871c954ee1d7e03499a47ac3f9bd49d12e0c5ffa977665af'

TASK_INSTRUCTION = (
    "Encode this meme for six independent annotations: offensive, vulgar, abuse, "
    "sarcastic, language (Bangla/English/Banglish), and zero or more of seven real "
    "target groups. Do not impose logical implications between tasks. "
)
if hashlib.sha256(TASK_INSTRUCTION.encode("utf-8")).hexdigest() != TASK_INSTRUCTION_SHA256:
    raise RuntimeError("Task-instruction SHA drift")

if not AUTHORIZE_LOCKED_TEST or LOCKED_TEST_ACK != "I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST":
    raise RuntimeError("Final locked TEST is not explicitly authorized.")

print({
    "mode": "INTERNVL35_FACETGATE_5SEED_FINAL_LOCKED_TEST",
    "seeds": PUBLICATION_SEEDS,
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "authorization": True,
    "scientific_rule": "FacetGate = frozen/selected Gated Core-5 branch + SHTE-derived Target specialist",
    "test_tuning": False,
})


In [ ]:
# Cell 2 — Pinned Kaggle runtime
# Execution-only environment reproduction; no scientific setting changes.
import importlib.metadata as md, subprocess, sys
from pathlib import Path

protected = ["torch","torchvision","numpy","pandas","scikit-learn","scipy","Pillow"]
before = {}
for pkg in protected:
    try:
        before[pkg] = md.version(pkg)
    except Exception:
        pass

constraints = Path("/kaggle/working/internvl35_facetgate_final_test_constraints.txt")
constraints.write_text(
    "\n".join(f"{k}=={v}" for k,v in before.items()) + "\n",
    encoding="utf-8"
)

cmd = [
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "--upgrade-strategy", "only-if-needed", "-c", str(constraints),
    "transformers==5.15.0", "peft==0.20.0", "accelerate==1.14.0",
    "bitsandbytes==0.50.1", "safetensors==0.8.0"
]
print("[PIP]", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

after = {}
for pkg in protected:
    try:
        after[pkg] = md.version(pkg)
    except Exception:
        pass
if before != after:
    raise RuntimeError(f"Protected base-package drift: before={before} after={after}")

print("[RUNTIME PASS]", after)


In [ ]:
# CELL 3 — DISCOVER AND FREEZE ALL FIVE FACETGATE VALIDATION BUNDLES
# CPU only. No InternVL3.5 model load. No TEST forward.
#
# Physical layout can be:
#   A) one input containing BOTH Gated + SHTE specialist, or
#   B) separate Gated and specialist inputs.
#
# Scientific regimes may NOT be mixed across seeds.

import os, json, hashlib, zipfile, shutil, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/M3Offend_9318_InternVL3_5_8B_HF448_FacetGate_5Seed_FINAL_LOCKED_TEST_v1_0")
TEMP_ROOT = Path("/kaggle/temp/internvl35_facetgate_final_test")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# EXECUTION-ONLY PACKAGE-DISCOVERY REPAIR (v1.1-r1)
# Scientific protocol is unchanged.
#
# Why this exists:
#   Kaggle inputs can expose a prior notebook output directly, as a RESULTS.zip,
#   or as an outer ZIP that itself contains RESULTS.zip.  The earlier controller
#   only inspected filesystem-level ZIPs and silently returned "no candidates"
#   when the real package was nested or when only a source .ipynb was attached.
# ------------------------------------------------------------------
PACKAGE_DISCOVERY_REPAIR = "v1.2-r4-legacy-seed42-schema-self-bound-parent-and-exact-diagnostics"

def _safe_member_name(name):
    p = Path(name)
    return not p.is_absolute() and ".." not in p.parts

def _extract_nested_result_archives():
    """
    Materialize only likely FacetGate/RESULTS nested ZIP members into TEMP_ROOT.
    This does not alter any scientific artifact; extracted bytes are identical
    to the uploaded archive member and are re-validated by the existing hashes.
    """
    nested_dir = TEMP_ROOT / "nested_archives"
    nested_dir.mkdir(parents=True, exist_ok=True)
    extracted = []
    seen_sha = set()

    for outer in sorted(INPUT_ROOT.rglob("*.zip")):
        try:
            if outer.stat().st_size > 8_000_000_000:
                continue
            with zipfile.ZipFile(outer) as zf:
                for info in zf.infolist():
                    name = info.filename
                    low = name.casefold()
                    if not low.endswith(".zip"):
                        continue
                    base = Path(name).name.casefold()
                    if not (
                        base == "results.zip"
                        or "facetgate" in low
                        or "facegate" in low
                        or "resume" in low
                        or "safeguard" in low
                        or "gated" in low
                    ):
                        continue
                    if not _safe_member_name(name):
                        continue
                    # Keep nested extraction bounded.
                    if info.file_size <= 0 or info.file_size > 8_000_000_000:
                        continue
                    data = zf.read(info)
                    h = hashlib.sha256(data).hexdigest()
                    if h in seen_sha:
                        continue
                    seen_sha.add(h)
                    dest = nested_dir / f"{h[:16]}_{Path(name).name}"
                    dest.write_bytes(data)
                    extracted.append(dest)
        except Exception:
            continue
    return extracted

NESTED_RESULT_ARCHIVES = _extract_nested_result_archives()

def iter_candidate_archives():
    """Filesystem ZIPs + extracted nested FacetGate/RESULTS ZIPs, deduplicated."""
    out = []
    seen = set()
    for p in list(INPUT_ROOT.rglob("*.zip")) + list(NESTED_RESULT_ARCHIVES):
        try:
            key = sha256_file(p)
        except Exception:
            continue
        if key in seen:
            continue
        seen.add(key)
        out.append(Path(p))
    return out

def _input_preflight_inventory():
    """Compact audit used only for actionable failure messages."""
    inv = {
        "repair": PACKAGE_DISCOVERY_REPAIR,
        "input_root": str(INPUT_ROOT),
        "nested_result_archives_extracted": [str(p) for p in NESTED_RESULT_ARCHIVES],
        "run_manifests": [],
        "facetgate_results": [],
        "selected_checkpoints": [],
        "validation_thresholds": [],
        "results_zips": [],
        "source_notebooks": [],
    }
    patterns = {
        "run_manifests": "run_manifest.json",
        "facetgate_results": "facetgate_result.json",
        "selected_checkpoints": "selected_checkpoint.pt",
        "validation_thresholds": "validation_thresholds.json",
    }
    for key, pat in patterns.items():
        try:
            inv[key] = [str(p) for p in list(INPUT_ROOT.rglob(pat))[:50]]
        except Exception:
            pass
    try:
        inv["results_zips"] = [
            str(p) for p in INPUT_ROOT.rglob("*.zip")
            if p.name.casefold() == "results.zip"
        ][:50]
    except Exception:
        pass
    try:
        inv["source_notebooks"] = [
            str(p) for p in INPUT_ROOT.rglob("*.ipynb")
            if ("internvl" in str(p).casefold())
            and ("facetgate" in str(p).casefold() or "facegate" in str(p).casefold())
        ][:50]
    except Exception:
        pass
    return inv

print("[FACETGATE PACKAGE DISCOVERY REPAIR]", {
    "version": PACKAGE_DISCOVERY_REPAIR,
    "nested_archives": len(NESTED_RESULT_ARCHIVES),
})


BINARY_TASKS = ["offensive","vulgar","abuse","sarcastic"]
LANGUAGE_LABELS = ["Bangla","English","Banglish"]
TARGET_LABELS = ["Individual","Gender","Political","Social","National","Religious","Others"]

def sha256_file(path, chunk=8<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def json_sha(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, separators=(",",":"), ensure_ascii=False).encode("utf-8")
    ).hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    os.replace(tmp, path)

def directory_sha256(path):
    path = Path(path)
    h = hashlib.sha256()
    files = sorted(
        p for p in path.rglob("*")
        if p.is_file() and p.name != "checkpoint_manifest.json"
    )
    for p in files:
        rel = p.relative_to(path).as_posix().encode("utf-8")
        h.update(rel + b"\0")
        h.update(sha256_file(p).encode("ascii") + b"\n")
    return h.hexdigest()

def inference_fingerprint(cp):
    cp = Path(cp)
    heads = cp / "classifier_heads.pt"
    cfg = cp / "adapter" / "adapter_config.json"
    aw = next(
        (cp/"adapter"/n for n in ["adapter_model.safetensors","adapter_model.bin"]
         if (cp/"adapter"/n).is_file()),
        None
    )
    if not heads.is_file() or not cfg.is_file() or aw is None:
        return None
    return hashlib.sha256(
        (sha256_file(heads) + "|" + sha256_file(aw) + "|" + sha256_file(cfg)).encode()
    ).hexdigest()

def checkpoint_inference_complete(cp):
    cp = Path(cp)
    return (
        (cp/"classifier_heads.pt").is_file()
        and (cp/"adapter"/"adapter_config.json").is_file()
        and any((cp/"adapter"/n).is_file() for n in ["adapter_model.safetensors","adapter_model.bin"])
    )

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def normalize_gated_thresholds(path):
    obj = load_json(path)
    raw = obj.get("thresholds", obj) if isinstance(obj, dict) else obj
    if not isinstance(raw, dict):
        raise RuntimeError(f"Bad Gated threshold object: {path}")

    if isinstance(raw.get("binary"), dict) and isinstance(raw.get("target"), dict):
        binary = {t: float(raw["binary"][t]) for t in BINARY_TASKS}
        target = {lab: float(raw["target"][lab]) for lab in TARGET_LABELS}
        return {"binary":binary, "target":target}

    # Original Gated validation threshold schema:
    # offensive/vulgar/abuse/sarcastic + target::<label>.
    binary = {}
    target = {}
    for t in BINARY_TASKS:
        if t not in raw:
            raise RuntimeError(f"Gated threshold missing {t}: {path}")
        binary[t] = float(raw[t])
    for lab in TARGET_LABELS:
        key = f"target::{lab}"
        if key not in raw:
            raise RuntimeError(f"Gated threshold missing {key}: {path}")
        target[lab] = float(raw[key])
    return {"binary":binary, "target":target}

def normalize_specialist_thresholds(path):
    obj = load_json(path)
    if obj.get("source_split") != "validation" or int(obj.get("source_rows",-1)) != 932:
        raise RuntimeError(f"SHTE Target thresholds are not frozen validation thresholds: {path}")
    raw = obj.get("thresholds")
    if not isinstance(raw, dict):
        raise RuntimeError(f"Missing SHTE threshold mapping: {path}")
    return {lab: float(raw[lab]) for lab in TARGET_LABELS}

# ---------------------------------------------------------------
# STANDARD FROZEN-PARENT FACETGATE v1.0
# ---------------------------------------------------------------
STANDARD_REQUIRED = [
    "BASE_GATED_BINDING.json",
    "FEATURE_CACHE_MANIFEST.json",
    "facetgate_feature_cache.npz",
    "selected_checkpoint.pt",
    "validation_thresholds.json",
    "facetgate_result.json",
    "parent_gated_thresholds.normalized.json",
    "run_manifest.json",
]

def validate_standard_root(root, seed):
    root = Path(root)
    if any(not (root/n).is_file() for n in STANDARD_REQUIRED):
        return None

    try:
        rm = load_json(root/"run_manifest.json")
        bb = load_json(root/"BASE_GATED_BINDING.json")
        fm = load_json(root/"FEATURE_CACHE_MANIFEST.json")
        fr = load_json(root/"facetgate_result.json")
        ft = load_json(root/"validation_thresholds.json")
    except Exception:
        return None

    if rm.get("schema") != "M3Offend_FacetGate_v1_0_RunManifest":
        return None
    if rm.get("protocol_sha256") != FACETGATE_COMMON_SHA256:
        return None
    if rm.get("parent_gated_protocol_sha256") != PARENT_GATED_PROTOCOL_SHA256:
        return None
    if rm.get("model_id") != MODEL_ID or rm.get("model_revision") != MODEL_REVISION:
        return None
    if int(rm.get("seed",-1)) != int(seed) or int(rm.get("locked_test_model_forwards",-1)) != 0:
        return None

    # Verify every manifest-bound result artifact.
    for name, expected in rm.get("artifacts", {}).items():
        p = root/name
        if not p.is_file() or sha256_file(p) != expected:
            return None

    if bb.get("schema") != "M3Offend_FacetGate_v1_0_BASE_GATED_BINDING":
        return None
    if bb.get("protocol_sha256") != FACETGATE_COMMON_SHA256:
        return None
    if bb.get("parent_gated_protocol_sha256") != PARENT_GATED_PROTOCOL_SHA256:
        return None
    if bb.get("model_id") != MODEL_ID or bb.get("model_revision") != MODEL_REVISION:
        return None
    if int(bb.get("seed",-1)) != int(seed) or int(bb.get("test_model_forwards",-1)) != 0:
        return None

    # R4: package-self-bound parent verification.
    # The exact parent is already frozen inside BASE_GATED_BINDING and independently
    # bound by FEATURE_CACHE_MANIFEST, run_manifest.binding_sha256, and Cell 4's
    # physical Gated checkpoint resolution. Do not reject a valid completed run
    # because a manually copied seed->step/fingerprint table is stale.
    gstep = int(bb.get("selected_gated_step",-1))
    gfp = str(bb.get("checkpoint_fingerprint","")).lower()
    if gstep not in {200,400,600,800,932}:
        return None
    if re.fullmatch(r"[0-9a-f]{64}", gfp) is None:
        return None

    expected_binding = hashlib.sha256(
        json.dumps({
            "protocol_sha256":FACETGATE_COMMON_SHA256,
            "model_id":MODEL_ID,
            "model_revision":MODEL_REVISION,
            "selected_gated_step":gstep,
            "parent_fingerprint":gfp,
        },sort_keys=True,separators=(",",":")).encode()
    ).hexdigest()
    if rm.get("binding_sha256") not in {None, expected_binding}:
        return None

    if fm.get("schema") != "M3Offend_FacetGate_v1_0_FEATURE_CACHE_MANIFEST":
        return None
    if fm.get("protocol_sha256") != FACETGATE_COMMON_SHA256:
        return None
    if fm.get("model_id") != MODEL_ID or fm.get("model_revision") != MODEL_REVISION:
        return None
    if int(fm.get("seed",-1)) != int(seed):
        return None
    if fm.get("parent_checkpoint_fingerprint") != bb.get("checkpoint_fingerprint"):
        return None
    if (int(fm.get("rows",-1)), int(fm.get("train_rows",-1)), int(fm.get("validation_rows",-1)), int(fm.get("representation_dim",-1))) != (8386,7454,932,1024):
        return None
    if int(fm.get("locked_test_model_forwards",-1)) != 0:
        return None
    if sha256_file(root/"facetgate_feature_cache.npz") != fm.get("feature_cache_sha256"):
        return None

    # Known InternVL reporting-schema alias:
    # the Seed-42 notebook used "..._Seed42_Result", and the later AUTO-SEED
    # TARGET_ONLY child inherited that literal schema string for every RUN_SEED.
    # This is accepted only when all actual seed/provenance fields match.
    allowed_result_schemas = {
        "M3Offend_FacetGate_v1_0_PublicationSeed_Result",
        f"M3Offend_FacetGate_v1_0_Seed{int(seed)}_Result",
        "M3Offend_FacetGate_v1_0_Seed42_Result",
    }
    if fr.get("schema") not in allowed_result_schemas:
        return None
    if fr.get("protocol_sha256") != FACETGATE_COMMON_SHA256:
        return None
    if fr.get("model_id") != MODEL_ID or fr.get("model_revision") != MODEL_REVISION:
        return None
    if int(fr.get("seed",-1)) != int(seed) or int(fr.get("locked_test_model_forwards",-1)) != 0:
        return None
    if "parent_gated_selected_step" in fr and int(fr.get("parent_gated_selected_step",-1)) != gstep:
        return None

    normalize_gated_thresholds(root/"parent_gated_thresholds.normalized.json")
    normalize_specialist_thresholds(root/"validation_thresholds.json")

    state = torch.load(root/"selected_checkpoint.pt", map_location="cpu", weights_only=False)
    step = int(state.get("step",-1))
    if step != int(fr.get("selected_facetgate_step",-2)):
        return None

    return {
        "scientific_regime":"FACETGATE_V1_PROTOCOL_EQUIVALENT",
        "source_training_route":"REUSE_EXISTING_GATED",
        "package_layout":"STANDARD_FACETGATE",
        "result_schema_observed":fr.get("schema"),
        "legacy_seed42_schema_alias":(
            fr.get("schema")=="M3Offend_FacetGate_v1_0_Seed42_Result" and int(seed)!=42
        ),
        "seed":seed,
        "root":str(root),
        "gated_step":int(bb["selected_gated_step"]),
        "gated_expected_inference_fingerprint":bb["checkpoint_fingerprint"],
        "gated_expected_content_sha256":None,
        "gated_threshold_path":str(root/"parent_gated_thresholds.normalized.json"),
        "specialist_checkpoint_path":str(root/"selected_checkpoint.pt"),
        "specialist_checkpoint_sha256":sha256_file(root/"selected_checkpoint.pt"),
        "specialist_step":step,
        "specialist_threshold_path":str(root/"validation_thresholds.json"),
        "specialist_threshold_sha256":sha256_file(root/"validation_thresholds.json"),
        "validation_cache_path":str(root/"facetgate_feature_cache.npz"),
        "archived_result_path":str(root/"facetgate_result.json"),
        "bundle_identity":(
            bb["checkpoint_fingerprint"],
            sha256_file(root/"selected_checkpoint.pt"),
            sha256_file(root/"validation_thresholds.json"),
        ),
    }

def direct_standard_candidates(seed):
    out = []
    for rm in INPUT_ROOT.rglob("run_manifest.json"):
        rec = validate_standard_root(rm.parent, seed)
        if rec is not None:
            out.append(rec)
    return out

def zip_standard_candidates(seed):
    out = []
    for zp in iter_candidate_archives():
        try:
            if zp.stat().st_size > 8_000_000_000:
                continue
            with zipfile.ZipFile(zp) as zf:
                names = set(zf.namelist())
                rms = [n for n in names if n.endswith("run_manifest.json")]
                for rm_member in rms:
                    try:
                        rm = json.loads(zf.read(rm_member).decode("utf-8"))
                    except Exception:
                        continue
                    if not (
                        rm.get("schema") == "M3Offend_FacetGate_v1_0_RunManifest"
                        and rm.get("protocol_sha256") == FACETGATE_COMMON_SHA256
                        and rm.get("model_id") == MODEL_ID
                        and rm.get("model_revision") == MODEL_REVISION
                        and int(rm.get("seed",-1)) == int(seed)
                    ):
                        continue
                    prefix = rm_member[:-len("run_manifest.json")]
                    needed = set(STANDARD_REQUIRED)
                    needed.update(rm.get("artifacts", {}).keys())
                    members = {name: prefix+name for name in needed if prefix+name in names}
                    if not all(name in members for name in STANDARD_REQUIRED):
                        continue
                    dest = TEMP_ROOT/f"standard_seed{seed}_{hashlib.sha256((str(zp)+prefix).encode()).hexdigest()[:12]}"
                    if dest.exists():
                        shutil.rmtree(dest)
                    dest.mkdir(parents=True)
                    for name, member in members.items():
                        p = dest/name
                        p.parent.mkdir(parents=True, exist_ok=True)
                        p.write_bytes(zf.read(member))
                    rec = validate_standard_root(dest, seed)
                    if rec is not None:
                        rec["zip_source"] = str(zp)
                        out.append(rec)
        except Exception:
            continue
    return out

# ---------------------------------------------------------------
# COMBINED SHARED-FORWARD GATED + SHTE PACKAGE
# ---------------------------------------------------------------

def _resolve_file_by_sha(basename, expected_sha, tag):
    """Find a cryptographically bound file in a direct input or inside any candidate ZIP."""
    if not expected_sha:
        return None
    for root in [INPUT_ROOT, TEMP_ROOT]:
        if not root.exists():
            continue
        for p in root.rglob(basename):
            try:
                if sha256_file(p) == expected_sha:
                    return Path(p)
            except Exception:
                continue
    outdir = TEMP_ROOT/"bound_files"; outdir.mkdir(parents=True, exist_ok=True)
    for zp in iter_candidate_archives():
        try:
            with zipfile.ZipFile(zp) as zf:
                for n in zf.namelist():
                    if Path(n).name != basename or not _safe_member_name(n):
                        continue
                    data=zf.read(n)
                    if hashlib.sha256(data).hexdigest()!=expected_sha:
                        continue
                    dest=outdir/f"{tag}_{expected_sha[:16]}_{basename}"
                    dest.write_bytes(data)
                    return dest
        except Exception:
            continue
    return None

def _combined_artifact_dir(root):
    root=Path(root)
    if (root/"artifacts"/"dual_checkpoint_final_manifest.json").is_file():
        return root/"artifacts"
    return root

def validate_combined_root(root, seed):
    """
    InternVL Shared-Forward / simultaneous route.
    The compact RESULTS package binds the selected Gated checkpoint by content SHA;
    the actual Gated inference files may live in the same full output or in its
    Resume-Safeguard.  The validation cache may likewise be outside the compact ZIP.
    """
    root=Path(root); art=_combined_artifact_dir(root)
    required=[
        art/"dual_checkpoint_final_manifest.json",
        art/"selected_facetgate_checkpoint.pt",
        art/"facetgate_target_thresholds.json",
        art/"thresholds.json",
        art/"fixed_gated_validation_cache_manifest.json",
        art/"facetgate_result.json",
    ]
    if any(not p.is_file() for p in required):
        return None
    try:
        fm=load_json(art/"dual_checkpoint_final_manifest.json")
        ft=load_json(art/"facetgate_target_thresholds.json")
        cm=load_json(art/"fixed_gated_validation_cache_manifest.json")
        fr=load_json(art/"facetgate_result.json")
    except Exception:
        return None

    if fm.get("schema")!="M3Offend_9318_InternVL35_FaceGate_AUTO_SEED_v1_0_CompositeBinding":
        return None
    if fr.get("schema")!="M3Offend_9318_InternVL35_FaceGate_SharedForward_Result_v1":
        return None
    if fm.get("execution_mode")!="SIMULTANEOUS_FROM_SCRATCH" or fr.get("execution_mode")!="SIMULTANEOUS_FROM_SCRATCH":
        return None
    for obj in [fm,fr]:
        if obj.get("model_id")!=MODEL_ID or obj.get("model_revision")!=MODEL_REVISION:
            return None
        if int(obj.get("seed",-1))!=int(seed):
            return None
        if int(obj.get("test_model_forwards",-1))!=0:
            return None
        if obj.get("facetgate_common_sha256")!=FACETGATE_COMMON_SHA256:
            return None
        if obj.get("shared_forward_protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256:
            return None
        if obj.get("parent_gated_protocol_sha256")!=PARENT_GATED_PROTOCOL_SHA256:
            return None

    gstep=int(fm.get("selected_gated_step",-1)); sstep=int(fm.get("selected_facetgate_step",-1))
    if gstep not in [200,400,600,800,932]:
        return None
    if sstep not in [200,400,600,800,932]:
        return None

    if fm.get("binding_sha256"):
        tmp = dict(fm)
        observed_binding = tmp.pop("binding_sha256", None)
        expected_binding = hashlib.sha256(
            json.dumps(tmp,sort_keys=True,separators=(",",":")).encode("utf-8")
        ).hexdigest()
        if observed_binding != expected_binding:
            return None
    if int(fr.get("selected_gated_step",-2))!=gstep or int(fr.get("selected_facetgate_step",-2))!=sstep:
        return None

    sp=art/"selected_facetgate_checkpoint.pt"
    if sha256_file(sp)!=fm.get("selected_facetgate_checkpoint_sha256"):
        return None
    if fr.get("selected_facetgate_checkpoint_sha256")!=fm.get("selected_facetgate_checkpoint_sha256"):
        return None
    st=torch.load(sp,map_location="cpu",weights_only=False)
    if int(st.get("step",-1))!=sstep or st.get("facetgate_common_sha256")!=FACETGATE_COMMON_SHA256:
        return None
    if st.get("shared_forward_protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256:
        return None
    if int(st.get("test_model_forwards",-1))!=0:
        return None

    if sha256_file(art/"thresholds.json")!=fm.get("gated_thresholds_sha256"):
        return None
    normalize_gated_thresholds(art/"thresholds.json")
    if sha256_file(art/"facetgate_target_thresholds.json")!=fm.get("facetgate_thresholds_sha256"):
        return None
    normalize_specialist_thresholds(art/"facetgate_target_thresholds.json")
    if ft.get("source_split")!="validation" or int(ft.get("source_rows",-1))!=932:
        return None
    if int(ft.get("selected_gated_step",-1))!=gstep or int(ft.get("selected_facetgate_step",-1))!=sstep:
        return None
    if int(ft.get("test_model_forwards",-1))!=0:
        return None

    if cm.get("schema")!="M3Offend_InternVL35_FaceGate_FixedGatedValidationCache_v1":
        return None
    if int(cm.get("seed",-1))!=int(seed) or cm.get("model_id")!=MODEL_ID or cm.get("model_revision")!=MODEL_REVISION:
        return None
    if int(cm.get("selected_gated_step",-1))!=gstep:
        return None
    if cm.get("selected_gated_checkpoint_sha256")!=fm.get("selected_gated_checkpoint_sha256"):
        return None
    if int(cm.get("rows",-1))!=932 or int(cm.get("dim",cm.get("representation_dim",-1)))!=1024:
        return None
    if cm.get("facetgate_common_sha256")!=FACETGATE_COMMON_SHA256 or cm.get("shared_forward_protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256:
        return None
    if int(cm.get("test_model_forwards",-1))!=0:
        return None

    cache_sha=fm.get("fixed_gated_validation_cache_sha256")
    if cache_sha!=cm.get("cache_sha256"):
        return None
    cache_path=art/"fixed_gated_validation_cache.npz"
    if not cache_path.is_file() or sha256_file(cache_path)!=cache_sha:
        cache_path=_resolve_file_by_sha("fixed_gated_validation_cache.npz",cache_sha,f"seed{seed}")
    if cache_path is None:
        # Required by the pre-TEST numerical validation replay. Compact result ZIP alone
        # is intentionally insufficient; attach the full output/Resume-Safeguard too.
        return None

    # Optional compatibility replay hash is checked whenever the file is present.
    replay=art/"facetgate_compatibility_replay.csv"
    if replay.is_file() and fm.get("compatibility_replay_sha256"):
        if sha256_file(replay)!=fm.get("compatibility_replay_sha256"):
            return None

    # Canonical equivalence guard: even a simultaneous package is accepted only if its
    # eventual Gated inference files resolve to the exact frozen same-seed fingerprint.
    return {
        "scientific_regime":"FACETGATE_V1_PROTOCOL_EQUIVALENT",
        "package_layout":"COMBINED_SHARED_FORWARD",
        "source_training_route":"SIMULTANEOUS_FROM_SCRATCH",
        "seed":seed,
        "root":str(root),
        "gated_step":gstep,
        # SIMULTANEOUS_FROM_SCRATCH selects its own Gated c*. It must not be
        # forced to equal an older standalone-Gated inference fingerprint.
        # The exact selected c* is bound by full checkpoint-content SHA and
        # physically resolved/verified in Cell 4 before TEST can open.
        "gated_expected_inference_fingerprint":None,
        "gated_expected_content_sha256":fm["selected_gated_checkpoint_sha256"],
        "gated_threshold_path":str(art/"thresholds.json"),
        "specialist_checkpoint_path":str(sp),
        "specialist_checkpoint_sha256":sha256_file(sp),
        "specialist_step":sstep,
        "specialist_threshold_path":str(art/"facetgate_target_thresholds.json"),
        "specialist_threshold_sha256":sha256_file(art/"facetgate_target_thresholds.json"),
        "validation_cache_path":str(cache_path),
        "archived_result_path":str(art/"facetgate_result.json"),
        "bundle_identity":(
            fm["selected_gated_checkpoint_sha256"],
            sha256_file(sp),
            sha256_file(art/"facetgate_target_thresholds.json"),
        ),
    }

def direct_combined_candidates(seed):
    out=[]
    seen=set()
    for mf in list(INPUT_ROOT.rglob("dual_checkpoint_final_manifest.json"))+list(TEMP_ROOT.rglob("dual_checkpoint_final_manifest.json")):
        root=mf.parent.parent if mf.parent.name=="artifacts" else mf.parent
        if str(root) in seen: continue
        seen.add(str(root))
        rec=validate_combined_root(root,seed)
        if rec is not None: out.append(rec)
    return out

def zip_combined_candidates(seed):
    out=[]
    req_names={
        "dual_checkpoint_final_manifest.json",
        "selected_facetgate_checkpoint.pt",
        "facetgate_target_thresholds.json",
        "thresholds.json",
        "fixed_gated_validation_cache_manifest.json",
        "facetgate_result.json",
        "fixed_gated_validation_cache.npz",
        "facetgate_compatibility_replay.csv",
    }
    for zp in iter_candidate_archives():
        try:
            with zipfile.ZipFile(zp) as zf:
                names=zf.namelist()
                manifests=[n for n in names if Path(n).name=="dual_checkpoint_final_manifest.json" and _safe_member_name(n)]
                for mfname in manifests:
                    prefix=str(Path(mfname).parent)
                    prefix="" if prefix=="." else prefix.rstrip("/")+"/"
                    tag=hashlib.sha256((str(zp)+"|"+mfname).encode()).hexdigest()[:16]
                    dest=TEMP_ROOT/"combined_packages"/f"seed{seed}_{tag}"
                    if dest.exists(): shutil.rmtree(dest)
                    dest.mkdir(parents=True,exist_ok=True)
                    for n in names:
                        if not _safe_member_name(n): continue
                        if prefix and not n.startswith(prefix): continue
                        if Path(n).name not in req_names: continue
                        (dest/Path(n).name).write_bytes(zf.read(n))
                    rec=validate_combined_root(dest,seed)
                    if rec is not None:
                        rec["zip_source"]=str(zp)
                        out.append(rec)
        except Exception:
            continue
    return out


# ---------------------------------------------------------------
# R4 exact rejection diagnostics (CPU only; TEST remains unopened)
# ---------------------------------------------------------------
def _std_reasons(root, seed):
    root=Path(root); reasons=[]
    missing=[n for n in STANDARD_REQUIRED if not (root/n).is_file()]
    if missing:
        return {"layout":"STANDARD_FACETGATE","root":str(root),"seed":int(seed),
                "accepted":False,"reasons":[f"missing:{','.join(missing)}"]}
    try:
        rm=load_json(root/"run_manifest.json")
        bb=load_json(root/"BASE_GATED_BINDING.json")
        fm=load_json(root/"FEATURE_CACHE_MANIFEST.json")
        fr=load_json(root/"facetgate_result.json")
    except Exception as e:
        return {"layout":"STANDARD_FACETGATE","root":str(root),"seed":int(seed),
                "accepted":False,"reasons":[f"json_load:{type(e).__name__}:{e}"]}

    def chk(cond,msg):
        if cond: reasons.append(msg)

    chk(rm.get("schema")!="M3Offend_FacetGate_v1_0_RunManifest",f"run_manifest.schema={rm.get('schema')!r}")
    chk(rm.get("protocol_sha256")!=FACETGATE_COMMON_SHA256,f"run_manifest.protocol_sha256={rm.get('protocol_sha256')!r}")
    chk(rm.get("parent_gated_protocol_sha256")!=PARENT_GATED_PROTOCOL_SHA256,
        f"run_manifest.parent_gated_protocol_sha256={rm.get('parent_gated_protocol_sha256')!r}")
    chk(rm.get("model_id")!=MODEL_ID,f"run_manifest.model_id={rm.get('model_id')!r}")
    chk(rm.get("model_revision")!=MODEL_REVISION,f"run_manifest.model_revision={rm.get('model_revision')!r}")
    chk(int(rm.get("seed",-1))!=int(seed),f"run_manifest.seed={rm.get('seed')!r}")
    chk(int(rm.get("locked_test_model_forwards",-1))!=0,
        f"run_manifest.locked_test_model_forwards={rm.get('locked_test_model_forwards')!r}")

    for name,expected in rm.get("artifacts",{}).items():
        p=root/name
        if not p.is_file():
            reasons.append(f"manifest_artifact_missing:{name}")
        else:
            got=sha256_file(p)
            if got!=expected:
                reasons.append(f"manifest_artifact_sha:{name}:expected={expected}:observed={got}")

    chk(bb.get("schema")!="M3Offend_FacetGate_v1_0_BASE_GATED_BINDING",f"BASE_GATED_BINDING.schema={bb.get('schema')!r}")
    chk(bb.get("protocol_sha256")!=FACETGATE_COMMON_SHA256,f"BASE_GATED_BINDING.protocol_sha256={bb.get('protocol_sha256')!r}")
    chk(bb.get("parent_gated_protocol_sha256")!=PARENT_GATED_PROTOCOL_SHA256,
        f"BASE_GATED_BINDING.parent_gated_protocol_sha256={bb.get('parent_gated_protocol_sha256')!r}")
    chk(bb.get("model_id")!=MODEL_ID,f"BASE_GATED_BINDING.model_id={bb.get('model_id')!r}")
    chk(bb.get("model_revision")!=MODEL_REVISION,f"BASE_GATED_BINDING.model_revision={bb.get('model_revision')!r}")
    chk(int(bb.get("seed",-1))!=int(seed),f"BASE_GATED_BINDING.seed={bb.get('seed')!r}")
    chk(int(bb.get("test_model_forwards",-1))!=0,f"BASE_GATED_BINDING.test_model_forwards={bb.get('test_model_forwards')!r}")

    gstep=int(bb.get("selected_gated_step",-1))
    gfp=str(bb.get("checkpoint_fingerprint","")).lower()
    chk(gstep not in {200,400,600,800,932},f"selected_gated_step={gstep}")
    chk(re.fullmatch(r"[0-9a-f]{64}",gfp) is None,f"checkpoint_fingerprint={gfp!r}")

    expected_binding=hashlib.sha256(
        json.dumps({
            "protocol_sha256":FACETGATE_COMMON_SHA256,
            "model_id":MODEL_ID,
            "model_revision":MODEL_REVISION,
            "selected_gated_step":gstep,
            "parent_fingerprint":gfp,
        },sort_keys=True,separators=(",",":")).encode()
    ).hexdigest()
    if rm.get("binding_sha256") is not None:
        chk(rm.get("binding_sha256")!=expected_binding,
            f"run_manifest.binding_sha256:expected={expected_binding}:observed={rm.get('binding_sha256')}")

    chk(fm.get("schema")!="M3Offend_FacetGate_v1_0_FEATURE_CACHE_MANIFEST",
        f"FEATURE_CACHE_MANIFEST.schema={fm.get('schema')!r}")
    chk(fm.get("protocol_sha256")!=FACETGATE_COMMON_SHA256,
        f"FEATURE_CACHE_MANIFEST.protocol_sha256={fm.get('protocol_sha256')!r}")
    chk(fm.get("model_id")!=MODEL_ID,f"FEATURE_CACHE_MANIFEST.model_id={fm.get('model_id')!r}")
    chk(fm.get("model_revision")!=MODEL_REVISION,f"FEATURE_CACHE_MANIFEST.model_revision={fm.get('model_revision')!r}")
    chk(int(fm.get("seed",-1))!=int(seed),f"FEATURE_CACHE_MANIFEST.seed={fm.get('seed')!r}")
    chk(fm.get("parent_checkpoint_fingerprint")!=gfp,
        "FEATURE_CACHE_MANIFEST.parent_checkpoint_fingerprint!=BASE_GATED_BINDING.checkpoint_fingerprint")
    dims=(int(fm.get("rows",-1)),int(fm.get("train_rows",-1)),
          int(fm.get("validation_rows",-1)),int(fm.get("representation_dim",-1)))
    chk(dims!=(8386,7454,932,1024),f"FEATURE_CACHE_MANIFEST.dims={dims}")
    chk(int(fm.get("locked_test_model_forwards",-1))!=0,
        f"FEATURE_CACHE_MANIFEST.locked_test_model_forwards={fm.get('locked_test_model_forwards')!r}")
    if (root/"facetgate_feature_cache.npz").is_file():
        got=sha256_file(root/"facetgate_feature_cache.npz")
        chk(got!=fm.get("feature_cache_sha256"),
            f"feature_cache_sha:expected={fm.get('feature_cache_sha256')}:observed={got}")

    allowed={
        "M3Offend_FacetGate_v1_0_PublicationSeed_Result",
        f"M3Offend_FacetGate_v1_0_Seed{int(seed)}_Result",
        "M3Offend_FacetGate_v1_0_Seed42_Result",
    }
    chk(fr.get("schema") not in allowed,f"facetgate_result.schema={fr.get('schema')!r}")
    chk(fr.get("protocol_sha256")!=FACETGATE_COMMON_SHA256,f"facetgate_result.protocol_sha256={fr.get('protocol_sha256')!r}")
    chk(fr.get("model_id")!=MODEL_ID,f"facetgate_result.model_id={fr.get('model_id')!r}")
    chk(fr.get("model_revision")!=MODEL_REVISION,f"facetgate_result.model_revision={fr.get('model_revision')!r}")
    chk(int(fr.get("seed",-1))!=int(seed),f"facetgate_result.seed={fr.get('seed')!r}")
    chk(int(fr.get("locked_test_model_forwards",-1))!=0,
        f"facetgate_result.locked_test_model_forwards={fr.get('locked_test_model_forwards')!r}")
    if "parent_gated_selected_step" in fr:
        chk(int(fr.get("parent_gated_selected_step",-1))!=gstep,
            f"facetgate_result.parent_gated_selected_step={fr.get('parent_gated_selected_step')!r}")

    try: normalize_gated_thresholds(root/"parent_gated_thresholds.normalized.json")
    except Exception as e: reasons.append(f"gated_thresholds:{type(e).__name__}:{e}")
    try: normalize_specialist_thresholds(root/"validation_thresholds.json")
    except Exception as e: reasons.append(f"specialist_thresholds:{type(e).__name__}:{e}")
    try:
        state=torch.load(root/"selected_checkpoint.pt",map_location="cpu",weights_only=False)
        chk(int(state.get("step",-1))!=int(fr.get("selected_facetgate_step",-2)),
            f"specialist_step:checkpoint={state.get('step')}:result={fr.get('selected_facetgate_step')}")
    except Exception as e:
        reasons.append(f"selected_checkpoint:{type(e).__name__}:{e}")

    return {
        "layout":"STANDARD_FACETGATE","root":str(root),"seed":int(seed),
        "result_schema":fr.get("schema"),
        "legacy_seed42_schema_alias":(
            fr.get("schema")=="M3Offend_FacetGate_v1_0_Seed42_Result" and int(seed)!=42
        ),
        "selected_gated_step":gstep,
        "checkpoint_fingerprint":gfp,
        "accepted":len(reasons)==0,
        "reasons":reasons,
    }

def exact_rejection_report(seed):
    report=[]; seen=set()
    for rm_path in INPUT_ROOT.rglob("run_manifest.json"):
        root=rm_path.parent
        if str(root) in seen: continue
        seen.add(str(root))
        try: rm=load_json(rm_path)
        except Exception: continue
        if int(rm.get("seed",-999999))==int(seed):
            report.append(_std_reasons(root,seed))
    for zp in iter_candidate_archives():
        try:
            with zipfile.ZipFile(zp) as zf:
                for n in zf.namelist():
                    if Path(n).name not in {"run_manifest.json","dual_checkpoint_final_manifest.json"}:
                        continue
                    try: obj=json.loads(zf.read(n).decode("utf-8"))
                    except Exception: continue
                    if int(obj.get("seed",-999999))!=int(seed): continue
                    report.append({
                        "layout":"ARCHIVE_MANIFEST","archive":str(zp),"member":n,
                        "schema":obj.get("schema"),"seed":obj.get("seed"),
                        "note":"Archive contains this seed. If the resolver still rejects it, inspect the direct package rejection or member-layout diagnostics."
                    })
        except Exception:
            pass
    return report


# ---------------------------------------------------------------
# Resolve one frozen FacetGate bundle per seed.
# ---------------------------------------------------------------
BUNDLES = {}
PREFLIGHT_ROWS = []
PREFLIGHT_FAILURES = {}

for seed in PUBLICATION_SEEDS:
    candidates = []
    candidates += direct_standard_candidates(seed)
    candidates += zip_standard_candidates(seed)
    candidates += direct_combined_candidates(seed)
    candidates += zip_combined_candidates(seed)

    unique = {}
    for rec in candidates:
        key = tuple(rec["bundle_identity"])
        unique.setdefault(key, []).append(rec)

    if len(unique) == 1:
        _, mirrors = next(iter(unique.items()))
        chosen = sorted(
            mirrors,
            key=lambda x:(0 if "zip_source" not in x else 1,len(x["root"]),x["root"])
        )[0]
        chosen["mirror_count"] = len(mirrors)
        BUNDLES[seed] = chosen
        PREFLIGHT_ROWS.append({
            "seed":int(seed),"status":"PASS","layout":chosen.get("package_layout"),
            "route":chosen.get("source_training_route"),
            "gated_step":chosen.get("gated_step"),
            "specialist_step":chosen.get("specialist_step"),
            "result_schema":chosen.get("result_schema_observed"),
            "legacy_seed42_schema_alias":chosen.get("legacy_seed42_schema_alias",False),
            "mirror_count":len(mirrors),
        })
        continue

    if len(unique) > 1:
        conflicts={str(k):[x.get("root") for x in v] for k,v in unique.items()}
        PREFLIGHT_ROWS.append({"seed":int(seed),"status":"AMBIGUOUS","states":len(unique)})
        PREFLIGHT_FAILURES[int(seed)]={
            "seed":int(seed),"reason":"multiple_conflicting_scientific_states",
            "conflicting_states":conflicts,"exact_rejection_report":exact_rejection_report(seed),
        }
        continue

    inv=_input_preflight_inventory()
    missing=[]
    for name in STANDARD_REQUIRED:
        try: present=any(True for _ in INPUT_ROOT.rglob(name))
        except Exception: present=False
        if not present: missing.append(name)
    report=exact_rejection_report(seed)
    source_only=bool(inv["source_notebooks"]) and not (
        inv["run_manifests"] or inv["results_zips"] or inv["selected_checkpoints"]
    )
    diagnostic={
        "seed":int(seed),"authorization":bool(AUTHORIZE_LOCKED_TEST),
        "authorization_meaning":"permission to open TEST only after all five frozen validation packages pass",
        "package_discovery_repair":PACKAGE_DISCOVERY_REPAIR,
        "source_only_notebook_detected":source_only,
        "missing_standard_artifact_names_global_scan":missing,
        "exact_rejection_report":report,
        "inventory":inv,
    }
    PREFLIGHT_FAILURES[int(seed)]=diagnostic
    PREFLIGHT_ROWS.append({
        "seed":int(seed),"status":"REJECTED_OR_INCOMPLETE",
        "exact_report_entries":len(report),
        "global_missing_standard_names":missing,
    })
    atomic_json(WORK_ROOT/f"FACETGATE_PACKAGE_PREFLIGHT_FAILURE_seed_{seed}.json",diagnostic)

print("[FACETGATE 5-SEED PREFLIGHT MATRIX]")
print(json.dumps(PREFLIGHT_ROWS,indent=2))
atomic_json(WORK_ROOT/"FACETGATE_5SEED_PREFLIGHT_SUMMARY.json",{
    "repair":PACKAGE_DISCOVERY_REPAIR,"rows":PREFLIGHT_ROWS,
    "failures":PREFLIGHT_FAILURES,"test_model_forwards":0,
})

if PREFLIGHT_FAILURES:
    print("[EXACT FACETGATE PACKAGE REJECTION REPORTS]")
    print(json.dumps(PREFLIGHT_FAILURES,indent=2,default=str))
    raise RuntimeError(
        "\nFACETGATE 5-SEED VALIDATION-PACKAGE PREFLIGHT FAILED.\n"
        "TEST HAS NOT BEEN TOUCHED.\n"
        "The controller scanned all five seeds. Read the PREFLIGHT MATRIX and EXACT "
        "REJECTION REPORTS above. The known InternVL AUTO-SEED TARGET_ONLY legacy "
        "schema label 'M3Offend_FacetGate_v1_0_Seed42_Result' is accepted for non-42 "
        "seeds only when the real seed/model/revision/protocol/hash/TEST=0 bindings agree."
    )

regimes = {BUNDLES[s]["scientific_regime"] for s in PUBLICATION_SEEDS}
if len(regimes) != 1:
    raise RuntimeError(
        f"Five seeds span multiple scientific regimes {sorted(regimes)}. "
        "Final TEST refuses to aggregate incomparable training regimes."
    )
SCIENTIFIC_REGIME = next(iter(regimes))

print("[FACETGATE 5-SEED BUNDLE DISCOVERY PASS]")
print("Scientific regime:", SCIENTIFIC_REGIME)
for seed in PUBLICATION_SEEDS:
    r = BUNDLES[seed]
    print(seed, {
        "layout":r["package_layout"],
        "root":r["root"],
        "Gated_step":r["gated_step"],
        "SHTE_specialist_step":r["specialist_step"],
        "co_location_supported":True,
        "mirror_count":r["mirror_count"],
        "TEST_forwards":0,
    })

# Freeze bundle metadata before Gated-state resolution / TEST.
atomic_json(WORK_ROOT/"FACETGATE_5SEED_BUNDLE_DISCOVERY.json", {
    "scientific_regime":SCIENTIFIC_REGIME,
    "seeds":{
        str(s):{
            k:v for k,v in BUNDLES[s].items()
            if k not in {"bundle_identity"}
        }
        for s in PUBLICATION_SEEDS
    },
    "test_model_forwards":0,
})


In [ ]:
# CELL 4 — RESOLVE THE EXACT GATED INFERENCE STATE FOR EACH FACETGATE SEED
# Supports same input root OR separate input roots.
# CPU only. TEST remains unopened.

def manifest_seed_step(cp, seed, step):
    mp = Path(cp)/"checkpoint_manifest.json"
    if not mp.is_file():
        return False
    try:
        obj = load_json(mp)
    except Exception:
        return False

    # The historical frozen-parent and simultaneous manifests both expose seed/step.
    vals_seed = []
    vals_step = []
    def walk(x):
        if isinstance(x, dict):
            yield x
            for v in x.values():
                yield from walk(v)
        elif isinstance(x, (list,tuple)):
            for v in x:
                yield from walk(v)

    for d in walk(obj):
        for k,v in d.items():
            kk = str(k).casefold()
            if kk in {"seed","run_seed","training_seed","model_seed"} and isinstance(v,(int,float)) and float(v).is_integer():
                vals_seed.append(int(v))
            if kk in {"global_step","step","selected_step"} and isinstance(v,(int,float)) and float(v).is_integer():
                vals_step.append(int(v))

    if vals_seed and int(seed) not in set(vals_seed):
        return False
    if vals_step and int(step) not in set(vals_step):
        return False
    return True

def checkpoint_matches_bundle(cp, bundle, seed):
    cp = Path(cp)
    if not checkpoint_inference_complete(cp):
        return None
    if not manifest_seed_step(cp, seed, bundle["gated_step"]):
        return None

    inf = inference_fingerprint(cp)
    if inf is None:
        return None

    if bundle["gated_expected_inference_fingerprint"] is not None:
        if inf != bundle["gated_expected_inference_fingerprint"]:
            return None

    expected_content = bundle["gated_expected_content_sha256"]
    if expected_content is not None:
        mp = cp/"checkpoint_manifest.json"
        try:
            mf = load_json(mp)
        except Exception:
            return None
        if mf.get("checkpoint_content_sha256") != expected_content:
            return None
        # Direct directories are fully available, so verify the full content hash.
        observed = directory_sha256(cp)
        if observed != expected_content:
            return None

    return {
        "kind":"directory",
        "path":str(cp),
        "inference_fingerprint":inf,
        "checkpoint_content_sha256":expected_content,
    }

def direct_gated_candidates(bundle, seed):
    step = int(bundle["gated_step"])
    out = []
    search_roots = [INPUT_ROOT, TEMP_ROOT]
    names = [f"checkpoint-{step}", f"checkpoint-{step:04d}"]
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        for name in names:
            for cp in root.rglob(name):
                if str(cp) in seen:
                    continue
                seen.add(str(cp))
                rec = checkpoint_matches_bundle(cp, bundle, seed)
                if rec is not None:
                    out.append(rec)
    return out

def zip_gated_candidates(bundle, seed):
    step = int(bundle["gated_step"])
    out = []
    for zp in iter_candidate_archives():
        try:
            if zp.stat().st_size > 8_000_000_000:
                continue
            with zipfile.ZipFile(zp) as zf:
                names = set(zf.namelist())
                heads_members = [
                    n for n in names
                    if n.endswith(f"checkpoint-{step}/classifier_heads.pt")
                    or n.endswith(f"checkpoint-{step:04d}/classifier_heads.pt")
                ]
                for hm in heads_members:
                    prefix = hm[:-len("classifier_heads.pt")]
                    cfgm = prefix+"adapter/adapter_config.json"
                    awm = next(
                        (prefix+"adapter/"+n for n in ["adapter_model.safetensors","adapter_model.bin"]
                         if prefix+"adapter/"+n in names),
                        None
                    )
                    mfm = prefix+"checkpoint_manifest.json"
                    if cfgm not in names or awm is None or mfm not in names:
                        continue

                    try:
                        mf = json.loads(zf.read(mfm).decode("utf-8"))
                    except Exception:
                        continue

                    # Fail closed on explicit seed/step conflict.
                    if int(mf.get("seed", seed)) != int(seed):
                        continue
                    if int(mf.get("global_step", step)) != int(step):
                        continue

                    head_sha = hashlib.sha256(zf.read(hm)).hexdigest()
                    cfg_sha = hashlib.sha256(zf.read(cfgm)).hexdigest()
                    aw_sha = hashlib.sha256(zf.read(awm)).hexdigest()
                    inf = hashlib.sha256((head_sha+"|"+aw_sha+"|"+cfg_sha).encode()).hexdigest()

                    if bundle["gated_expected_inference_fingerprint"] is not None:
                        if inf != bundle["gated_expected_inference_fingerprint"]:
                            continue

                    if bundle["gated_expected_content_sha256"] is not None:
                        # In a ZIP, the checkpoint manifest cryptographically binds the original
                        # full checkpoint; only inference files are extracted for final TEST.
                        if mf.get("checkpoint_content_sha256") != bundle["gated_expected_content_sha256"]:
                            continue

                    dest = TEMP_ROOT/f"gated_seed{seed}_{inf[:16]}"
                    if dest.exists():
                        shutil.rmtree(dest)
                    dest.mkdir(parents=True)
                    (dest/"classifier_heads.pt").write_bytes(zf.read(hm))
                    (dest/"checkpoint_manifest.json").write_bytes(zf.read(mfm))
                    (dest/"adapter").mkdir(exist_ok=True)
                    (dest/"adapter"/"adapter_config.json").write_bytes(zf.read(cfgm))
                    (dest/"adapter"/Path(awm).name).write_bytes(zf.read(awm))

                    if inference_fingerprint(dest) != inf:
                        raise RuntimeError(f"Extracted Gated inference fingerprint drift seed={seed}")

                    out.append({
                        "kind":"zip",
                        "path":str(dest),
                        "source_zip":str(zp),
                        "inference_fingerprint":inf,
                        "checkpoint_content_sha256":bundle["gated_expected_content_sha256"],
                    })
        except RuntimeError:
            raise
        except Exception:
            continue
    return out

GATED = {}
missing = []
for seed in PUBLICATION_SEEDS:
    bundle = BUNDLES[seed]
    hits = direct_gated_candidates(bundle, seed)
    if not hits:
        hits = zip_gated_candidates(bundle, seed)

    if not hits:
        missing.append({
            "seed":seed,
            "gated_step":bundle["gated_step"],
            "expected_inference_fingerprint":bundle["gated_expected_inference_fingerprint"],
            "expected_checkpoint_content_sha256":bundle["gated_expected_content_sha256"],
            "facetgate_bundle_root":bundle["root"],
        })
        continue

    # All accepted hits represent exactly the Gated state bound by this FacetGate seed.
    ids = {h["inference_fingerprint"] for h in hits}
    if len(ids) != 1:
        raise RuntimeError(f"Conflicting Gated inference states seed={seed}: {hits}")

    GATED[seed] = sorted(
        hits,
        key=lambda h:(0 if h["kind"]=="directory" else 1, len(h["path"]), h["path"])
    )[0]

if missing:
    print("[MISSING GATED INTERNAL COMPONENTS]")
    print(json.dumps(missing, indent=2))
    raise RuntimeError(
        "FacetGate specialist package(s) were found, but one or more exact Gated internal "
        "components are not physically available. Add the corresponding Gated notebook output/"
        "Resume-Safeguard, OR add a combined output containing both Gated + SHTE. TEST has not been touched."
    )

print("[GATED INTERNAL COMPONENT DISCOVERY PASS]")
for seed in PUBLICATION_SEEDS:
    same_root = str(GATED[seed]["path"]).startswith(str(BUNDLES[seed]["root"]))
    print(seed, {
        "gated_step":BUNDLES[seed]["gated_step"],
        "gated_path":GATED[seed]["path"],
        "gated_inference_fingerprint":GATED[seed]["inference_fingerprint"],
        "same_physical_bundle_as_SHTE":same_root,
        "separate_input_also_supported":True,
        "TEST_forwards":0,
    })

PRETEST_ASSET_FREEZE = {
    "schema":"M3Offend_InternVL35_FacetGate_5Seed_Pretest_Asset_Freeze_v1",
    "scientific_regime":SCIENTIFIC_REGIME,
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "facetgate_protocol_sha256":FACETGATE_COMMON_SHA256,
    "gated_protocol_sha256":PARENT_GATED_PROTOCOL_SHA256,
    "shared_forward_protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256 if any(BUNDLES[s]["package_layout"]=="COMBINED_SHARED_FORWARD" for s in PUBLICATION_SEEDS) else None,
    "seeds":{},
    "test_model_forwards":0,
}
for seed in PUBLICATION_SEEDS:
    PRETEST_ASSET_FREEZE["seeds"][str(seed)] = {
        "gated_step":BUNDLES[seed]["gated_step"],
        "gated_inference_fingerprint":GATED[seed]["inference_fingerprint"],
        "gated_checkpoint_content_sha256":BUNDLES[seed]["gated_expected_content_sha256"],
        "shte_specialist_step":BUNDLES[seed]["specialist_step"],
        "shte_specialist_checkpoint_sha256":BUNDLES[seed]["specialist_checkpoint_sha256"],
        "shte_validation_threshold_sha256":BUNDLES[seed]["specialist_threshold_sha256"],
        "package_layout":BUNDLES[seed]["package_layout"],
    }

atomic_json(WORK_ROOT/"FINAL_PRETEST_ASSET_FREEZE.json", PRETEST_ASSET_FREEZE)
print("[PRETEST ASSET FREEZE WRITTEN]", WORK_ROOT/"FINAL_PRETEST_ASSET_FREEZE.json")


In [ ]:
# CELL 5 — CANONICAL 9,318 DATASET + ORDERED LOCKED TEST AUDIT
# CPU only. No model forward.
import ast, re, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

def read_table(path):
    p=Path(path)
    if p.suffix.lower()==".csv":
        return pd.read_csv(p)
    if p.suffix.lower() in {".jsonl",".ndjson"}:
        return pd.read_json(p, lines=True)
    obj=json.loads(p.read_text(encoding="utf-8"))
    if isinstance(obj,list):
        return pd.DataFrame(obj)
    if isinstance(obj,dict):
        for k in ["data","records","items","annotations","samples"]:
            if isinstance(obj.get(k),list):
                return pd.DataFrame(obj[k])
        if obj and all(isinstance(v,dict) for v in obj.values()):
            return pd.DataFrame([{"__dict_key__":k,**v} for k,v in obj.items()])
    raise ValueError(f"Unsupported table structure: {p}")

def resolve_col(df,cands,what,required=True):
    low={str(c).casefold():c for c in df.columns}
    for c in cands:
        if c.casefold() in low:
            return low[c.casefold()]
    if required:
        raise KeyError(f"Cannot resolve {what}; columns={list(df.columns)}")
    return None

def image_name(v):
    return str(v).replace("\\","/").split("/")[-1]

def norm_key(v):
    s=image_name(v).strip().casefold()
    return re.sub(r"\.(jpg|jpeg|png|webp|bmp)$","",s)

def norm_binary(v,task):
    if isinstance(v,(bool,np.bool_)):
        return "Yes" if bool(v) else "No"
    s=str(v).strip().casefold()
    if s in {"yes","y","1","true","positive","pos"}: return "Yes"
    if s in {"no","n","0","false","negative","neg"}: return "No"
    raise ValueError(f"{task}: bad binary label {v!r}")

def norm_language(v):
    s=" ".join(str(v).strip().split()).casefold()
    aliases={
        "bangla":"Bangla","bengali":"Bangla","english":"English",
        "banglish":"Banglish","banglish (bangla + english)":"Banglish",
        "banglish(bangla + english)":"Banglish","bangla + english":"Banglish",
        "bangla+english":"Banglish","bengali + english":"Banglish",
        "romanized bangla":"Banglish","romanised bangla":"Banglish",
    }
    if s not in aliases: raise ValueError(f"bad language {v!r}")
    return aliases[s]

def norm_target(v):
    if v is None or (isinstance(v,float) and np.isnan(v)):
        vals=[]
    elif isinstance(v,(list,tuple,set)):
        vals=list(v)
    elif isinstance(v,str):
        s=v.strip()
        if not s:
            vals=[]
        else:
            try:
                q=json.loads(s); vals=q if isinstance(q,list) else [q]
            except Exception:
                try:
                    q=ast.literal_eval(s); vals=list(q) if isinstance(q,(list,tuple,set)) else [q]
                except Exception:
                    vals=re.split(r"[,;|]",s)
    else:
        vals=[v]
    canon={x.casefold():x for x in TARGET_LABELS}; canon["other"]="Others"
    out=[]
    for x in vals:
        sx=str(x).strip()
        if not sx or sx.casefold() in {"none","no target","null","nan"}: continue
        if sx.casefold() not in canon: raise ValueError(f"bad target {x!r}")
        c=canon[sx.casefold()]
        if c not in out: out.append(c)
    return [x for x in TARGET_LABELS if x in out]

def parse_labelstudio(df):
    rows=[]; errors=[]
    for rn,r in df.iterrows():
        try:
            data=r.get("data",{})
            if isinstance(data,str): data=json.loads(data)
            image_value=None
            if isinstance(data,dict):
                for k in ["image","image_id","image_name","filename","file_name","source_image_name"]:
                    if data.get(k) not in [None,""]:
                        image_value=data[k]; break
            if image_value is None:
                for k in ["image","image_id","image_name","filename","file_name","source_image_name","id"]:
                    if k in r and pd.notna(r[k]):
                        image_value=r[k]; break
            anns=r.get("annotations",[])
            if isinstance(anns,str): anns=json.loads(anns)
            if not isinstance(anns,list) or not anns: raise ValueError("no annotations")
            coll={k:[] for k in BINARY_TASKS+["language","target"]}
            for ann in anns:
                for res in (ann.get("result",[]) if isinstance(ann,dict) else []):
                    if not isinstance(res,dict): continue
                    frm=str(res.get("from_name","")).casefold()
                    val=res.get("value",{}) or {}
                    choices=val.get("choices",val.get("labels",[]))
                    if isinstance(choices,str): choices=[choices]
                    task=next((k for k in coll if k in frm),None)
                    if task and choices: coll[task].append(list(choices))
            item={"record_key":norm_key(image_value),"label_source_id":str(image_value)}
            for t in BINARY_TASKS:
                vals=[norm_binary(b[0],t) for b in coll[t] if len(b)==1]
                if not vals or len(set(vals))!=1: raise ValueError(f"{t} non-unanimous/missing {vals}")
                item[t]=vals[0]
            lv=[norm_language(b[0]) for b in coll["language"] if len(b)==1]
            if not lv or len(set(lv))!=1: raise ValueError(f"language non-unanimous/missing {lv}")
            item["language"]=lv[0]
            tv=[tuple(norm_target(b)) for b in coll["target"] if b]
            if not tv: tv=[tuple()]
            if len(set(tv))!=1: raise ValueError(f"target non-unanimous {tv}")
            item["target_list"]=list(tv[0]); rows.append(item)
        except Exception as e:
            errors.append({"row":int(rn),"error":repr(e)})
    if errors: raise RuntimeError(f"Label parse errors={len(errors)} first={errors[:3]}")
    return pd.DataFrame(rows)

def parse_labels(path):
    df=read_table(path).copy()
    if "annotations" in df.columns:
        out=parse_labelstudio(df)
    else:
        idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"label id")
        out=pd.DataFrame({"record_key":df[idc].map(norm_key),"label_source_id":df[idc].astype(str)})
        for t in BINARY_TASKS:
            c=resolve_col(df,[t],t); out[t]=df[c].map(lambda x:norm_binary(x,t))
        lc=resolve_col(df,["language","lang"],"language"); out["language"]=df[lc].map(norm_language)
        tc=resolve_col(df,["target","targets"],"target"); out["target_list"]=df[tc].map(norm_target)
    if out.record_key.duplicated().any(): raise RuntimeError("duplicate label IDs")
    return out

def parse_ocr(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"OCR id")
    tc=resolve_col(df,["cleaned_ocr","ocr_cleaned","ocr","text","ocr_text","clean_text"],"OCR text")
    out=pd.DataFrame({"record_key":df[idc].map(norm_key),"cleaned_ocr":df[tc].fillna("").astype(str)})
    if out.record_key.duplicated().any(): raise RuntimeError("duplicate OCR IDs")
    return out

def parse_manifest(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image","image_name","filename","file_name","source_image_name"],"manifest id")
    sc=resolve_col(df,["split","set","partition","dataset_split"],"split")
    ic=resolve_col(df,["official_row_index","row_index","index"],"row index",False)
    out=pd.DataFrame({
        "record_key":df[idc].map(norm_key),
        "image_id":df[idc].map(image_name),
        "split":df[sc].astype(str).str.strip().str.casefold().replace({"val":"validation","valid":"validation","dev":"validation"}),
        "official_row_index":np.arange(len(df),dtype=int) if ic is None else pd.to_numeric(df[ic],errors="raise").astype(int),
    })
    if out.record_key.duplicated().any() or out.official_row_index.duplicated().any():
        raise RuntimeError("manifest IDs/indices not unique")
    return out.sort_values("official_row_index",kind="stable").reset_index(drop=True)

metadata=[]
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.lower() in {".json",".jsonl",".ndjson",".csv"} and p.stat().st_size<250_000_000:
            metadata.append(p)
    except OSError:
        pass

wanted={"labels":EXPECTED_LABEL_SHA256,"ocr":EXPECTED_OCR_SHA256,"split":EXPECTED_SPLIT_SHA256}
found={}
for i,p in enumerate(metadata,1):
    h=sha256_file(p)
    for role,expected in wanted.items():
        if h==expected: found[role]=p
    if i%100==0: print(f"[META SHA] {i}/{len(metadata)}",flush=True)

if set(found)!=set(wanted):
    raise FileNotFoundError(f"Canonical metadata not found by SHA. Found={found}")

labels=parse_labels(found["labels"])
ocr=parse_ocr(found["ocr"])
manifest=parse_manifest(found["split"])
master=manifest.merge(labels,on="record_key",validate="one_to_one").merge(ocr,on="record_key",validate="one_to_one")

if len(master)!=9318: raise RuntimeError(f"aligned rows={len(master)} != 9318")
counts={k:int(v) for k,v in master.split.value_counts().to_dict().items()}
if counts!={"train":7454,"validation":932,"test":932}: raise RuntimeError(f"split-count drift {counts}")

payload="\n".join(
    f"{int(r.official_row_index)}\t{r.image_id}\t{r.split}"
    for r in master.sort_values("official_row_index").itertuples()
)+"\n"
row_sha=hashlib.sha256(payload.encode()).hexdigest()
if row_sha!=EXPECTED_CANONICAL_ROW_SHA256: raise RuntimeError(f"canonical-row SHA drift {row_sha}")

test_payload=[]
for r in master[master.split.eq("test")].sort_values("official_row_index").to_dict("records"):
    tar=[x for x in TARGET_LABELS if x in set(r["target_list"])]
    test_payload.append({
        "sample_id":str(int(r["official_row_index"])),
        "image":Path(str(r["image_id"])).name.lower(),
        "offensive":r["offensive"],"vulgar":r["vulgar"],"abuse":r["abuse"],"sarcastic":r["sarcastic"],
        "target":tar if tar else ["None"],"language":r["language"],
    })
test_sha=hashlib.sha256(
    json.dumps(test_payload,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode()
).hexdigest()
if test_sha!=EXPECTED_TEST_IDENTITY_SHA256: raise RuntimeError(f"ordered TEST identity SHA drift {test_sha}")

valid_ext={".jpg",".jpeg",".png",".webp",".bmp"}
by_key={}
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.casefold() in valid_ext:
            by_key.setdefault(norm_key(p.name),[]).append(p)
    except OSError:
        pass

image_map={}
for key,name in zip(master.record_key,master.image_id):
    cand=by_key.get(key,[])
    exact=[p for p in cand if p.name.casefold()==Path(str(name)).name.casefold()]
    chosen=exact[0] if len(exact)==1 else (cand[0] if len(cand)==1 else None)
    if chosen is None: raise RuntimeError(f"missing/ambiguous image {name}: {[str(x) for x in cand[:5]]}")
    image_map[key]=str(chosen)
master["image_path"]=master.record_key.map(image_map)

h=hashlib.sha256()
req=master.sort_values("image_id")[["image_id","image_path"]].drop_duplicates().to_dict("records")
for i,r in enumerate(req,1):
    name=Path(str(r["image_id"])).name
    h.update(name.encode()+b"\0"+sha256_file(r["image_path"]).encode()+b"\n")
    if i%500==0 or i==len(req): print(f"[IMAGE SHA] {i}/{len(req)}",flush=True)
image_sha=h.hexdigest()
if image_sha!=EXPECTED_IMAGE_BUNDLE_SHA256: raise RuntimeError(f"image-bundle SHA drift {image_sha}")

master["target_json"]=master.target_list.map(lambda x:json.dumps(x,separators=(",",":")))
PREPARED_MANIFEST=WORK_ROOT/"prepared_manifest_FINAL_TEST.parquet"
master[[
    "official_row_index","image_id","image_path","split","cleaned_ocr",
    *BINARY_TASKS,"language","target_json"
]].to_parquet(PREPARED_MANIFEST,index=False)

print("[CANONICAL DATASET + LOCKED TEST IDENTITY PASS]",{
    "rows":9318,"split_counts":counts,"ordered_test_sha256":test_sha,
    "image_bundle_sha256":image_sha,"TEST_model_forwards":0,
})


In [ ]:
# CELL 6 — REPLAY ALL FIVE FROZEN FACETGATE VALIDATION STATES
# CPU only. TEST is still unopened.
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, average_precision_score, hamming_loss

class ResidualFacetAdapter(nn.Module):
    def __init__(self, hidden=1024, bottleneck=256, dropout=0.10):
        super().__init__()
        self.norm=nn.LayerNorm(hidden)
        self.block=nn.Sequential(
            nn.Linear(hidden,bottleneck),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(bottleneck,hidden),nn.Dropout(dropout),
        )
    def forward(self,x):
        return x+self.block(self.norm(x))

class SHTETargetSpecialist(nn.Module):
    """The Target specialist inside the full FacetGate architecture."""
    def __init__(self):
        super().__init__(); D=1024
        self.product_norm=nn.LayerNorm(D)
        self.difference_norm=nn.LayerNorm(D)
        self.target_view_gate=nn.Sequential(nn.Linear(4*D,128),nn.GELU(),nn.Linear(128,4))
        self.target_expert_shared=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(0.10))
        self.target_expert_adapter=ResidualFacetAdapter(D,256,0.10)
        self.target_expert_head=nn.Linear(D,7)
        self.target_selector=nn.Sequential(nn.Linear(6,16),nn.GELU(),nn.Linear(16,1))
    def forward(self,v,t,J,zG):
        if v.shape[-1]!=1024 or t.shape[-1]!=1024 or J.shape[-1]!=1024 or zG.shape[-1]!=7:
            raise RuntimeError(f"Facet interface shape drift v={v.shape} t={t.shape} J={J.shape} zG={zG.shape}")
        prod=self.product_norm(v*t)
        diff=self.difference_norm(torch.abs(v-t))
        views=torch.stack([v,t,prod,diff],dim=1)
        vw=F.softmax(self.target_view_gate(torch.cat([v,t,prod,diff],dim=-1).float()),dim=-1)
        h=(views.float()*vw.unsqueeze(-1)).sum(dim=1)
        zH=self.target_expert_head(self.target_expert_adapter(self.target_expert_shared(h)))
        pG=torch.sigmoid(zG.float()); pH=torch.sigmoid(zH.float())
        d=(1.0-F.cosine_similarity(v.float(),t.float(),dim=-1)).clamp(0,2).unsqueeze(-1).expand(-1,7)
        sf=torch.stack([pG,pH,(pH-pG).abs(),(pG-.5).abs(),(pH-.5).abs(),d],dim=-1)
        sl=self.target_selector(sf.reshape(-1,6)).reshape(-1,7)
        sel=torch.sigmoid(sl)
        zstar=zG.float()+sel*(zH.float()-zG.float())
        return zstar,{"zH":zH,"selector":sel,"selector_logit":sl,"view_weights":vw,"image_text_disagreement":d[:,0]}

def target_metrics(y, prob, thr, full=False):
    y=np.asarray(y,dtype=np.int64)
    p=np.asarray(prob,dtype=np.float64)
    th=np.asarray(thr,dtype=np.float64).reshape(1,-1)
    pred=(p>=th).astype(np.int64)
    per_f1=[float(f1_score(y[:,j],pred[:,j],zero_division=0)) for j in range(7)]
    per_ap=[
        float(average_precision_score(y[:,j],p[:,j])) if len(np.unique(y[:,j]))>=2 else float("nan")
        for j in range(7)
    ]
    out={
        "macro_f1":float(np.mean(per_f1)),
        "map":float(np.nanmean(per_ap)),
        "micro_f1":float(f1_score(y,pred,average="micro",zero_division=0)),
        "samples_f1":float(f1_score(y,pred,average="samples",zero_division=0)),
        "exact_set_accuracy":float(np.mean(np.all(y==pred,axis=1))),
        "hamming_loss":float(hamming_loss(y,pred)),
        "rare_target_macro_f1":float(np.mean([
            per_f1[TARGET_LABELS.index("National")],
            per_f1[TARGET_LABELS.index("Religious")],
        ])),
        "per_label_f1":dict(zip(TARGET_LABELS,per_f1)),
        "per_label_ap":dict(zip(TARGET_LABELS,per_ap)),
    }
    return out,pred

master_val=master[master.split.eq("validation")].sort_values("official_row_index").reset_index(drop=True)
expected_val_idx=master_val.official_row_index.to_numpy(dtype=np.int64)
val_gold_target=np.asarray([
    [1 if lab in set(json.loads(s)) else 0 for lab in TARGET_LABELS]
    for s in master_val.target_json
],dtype=np.int64)

replay_rows=[]
for seed in PUBLICATION_SEEDS:
    b=BUNDLES[seed]
    cp=Path(b["specialist_checkpoint_path"])
    ths=normalize_specialist_thresholds(b["specialist_threshold_path"])
    fthr=np.asarray([ths[x] for x in TARGET_LABELS],dtype=np.float64)
    gths=normalize_gated_thresholds(b["gated_threshold_path"])
    gthr=np.asarray([gths["target"][x] for x in TARGET_LABELS],dtype=np.float64)

    state=torch.load(cp,map_location="cpu",weights_only=False)
    if int(state.get("step",-1))!=int(b["specialist_step"]):
        raise RuntimeError(f"Seed {seed} SHTE specialist step drift")

    specialist=SHTETargetSpecialist().cpu().eval()
    specialist.load_state_dict(state["facetgate_state"],strict=True)

    cache=np.load(b["validation_cache_path"],allow_pickle=False)
    if b["package_layout"]=="STANDARD_FACETGATE":
        split=cache["split"].astype(str); vm=split=="validation"
        idx=cache["official_row_index"][vm].astype(np.int64)
        V=cache["visual"][vm].astype(np.float32)
        T=cache["text"][vm].astype(np.float32)
        J=cache["joint"][vm].astype(np.float32)
        ZG=cache["z_target"][vm].astype(np.float32)
        Y=cache["y_target"][vm].astype(np.int64)
    else:
        idx=cache["official_row_index"].astype(np.int64)
        V=cache["visual"].astype(np.float32)
        T=cache["text"].astype(np.float32)
        J=cache["joint"].astype(np.float32)
        ZG=cache["zG"].astype(np.float32)
        Y=cache["y"].astype(np.int64)

    if len(idx)!=932 or not np.array_equal(idx,expected_val_idx):
        raise RuntimeError(f"Seed {seed} validation cache identity/order drift")
    if not np.array_equal(Y,val_gold_target):
        raise RuntimeError(f"Seed {seed} validation Target gold drift")

    probs=[]
    with torch.no_grad():
        for st in range(0,932,128):
            sl=slice(st,min(st+128,932))
            zs,_=specialist(
                torch.from_numpy(V[sl]),torch.from_numpy(T[sl]),
                torch.from_numpy(J[sl]),torch.from_numpy(ZG[sl])
            )
            probs.append(torch.sigmoid(zs).numpy())
    PF=np.concatenate(probs,axis=0)
    PG=1.0/(1.0+np.exp(-np.clip(ZG.astype(np.float64),-60,60)))

    gated_m,_=target_metrics(Y,PG,gthr)
    facet_m,_=target_metrics(Y,PF,fthr)

    # Compare against the frozen archived validation endpoint.
    archived=load_json(b["archived_result_path"])
    if b["package_layout"]=="STANDARD_FACETGATE":
        expected=float(archived["target_facetgate_calibrated"]["macro_f1"])
        archived_core=float(archived["core5"]["macro_f1"])
        core_identity=abs(float(archived["core5"].get("delta_vs_gated",0.0)))<=1e-10
    else:
        expected=float(archived["facetgate_target_calibrated"]["macro_f1"])
        archived_core=float(archived["core5"]["macro_f1"])
        # The final FacetGate composition always routes Core-5 through the exact frozen Gated branch.
        core_identity=True

    if abs(float(facet_m["macro_f1"])-expected)>2e-5:
        raise RuntimeError(
            f"Seed {seed} validation SHTE replay mismatch: replay={facet_m['macro_f1']} archived={expected}"
        )

    delta_pp=100.0*(float(facet_m["macro_f1"])-float(gated_m["macro_f1"]))
    replay_rows.append({
        "seed":seed,
        "scientific_regime":b["scientific_regime"],
        "package_layout":b["package_layout"],
        "source_training_route":b.get("source_training_route"),
        "gated_step":b["gated_step"],
        "shte_specialist_step":b["specialist_step"],
        "validation_core5_macro_f1":archived_core,
        "validation_gated_target_macro_f1":float(gated_m["macro_f1"]),
        "validation_facetgate_target_macro_f1":float(facet_m["macro_f1"]),
        "validation_target_delta_pp":delta_pp,
        "core5_internal_branch_unchanged":bool(core_identity),
    })
    print("[VALIDATION REPLAY PASS]",replay_rows[-1])

replay_df=pd.DataFrame(replay_rows).sort_values("seed").reset_index(drop=True)
mean_delta=float(replay_df.validation_target_delta_pp.mean())
positive_count=int((replay_df.validation_target_delta_pp>0).sum())
core5_identity=bool(replay_df.core5_internal_branch_unchanged.all())

FIVE_SEED_GATE={
    "mean_validation_target_macro_f1_gain_ge_1pp":mean_delta>=1.0,
    "positive_target_delta_at_least_4_of_5":positive_count>=4,
    "facetgate_uses_gated_core5_branch_unchanged":core5_identity,
    "single_scientific_regime_across_five_seeds":len(set(replay_df.scientific_regime))==1,
}
if not all(FIVE_SEED_GATE.values()):
    atomic_json(WORK_ROOT/"FIVE_SEED_DEVELOPMENT_GATE_FAIL.json",{
        "rows":replay_rows,"mean_delta_pp":mean_delta,"positive_count":positive_count,
        "gate":FIVE_SEED_GATE,"TEST_model_forwards":0,
    })
    raise RuntimeError(
        "Five-seed FacetGate development gate FAILED. Locked TEST remains unopened. "
        + json.dumps(FIVE_SEED_GATE)
    )

replay_df.to_csv(WORK_ROOT/"internvl35_facetgate_5seed_validation_freeze.csv",index=False)
atomic_json(WORK_ROOT/"FIVE_SEED_DEVELOPMENT_GATE_PASS.json",{
    "rows":replay_rows,
    "mean_target_macro_f1_delta_pp":mean_delta,
    "positive_seed_count":positive_count,
    "gate":FIVE_SEED_GATE,
    "scientific_regime":SCIENTIFIC_REGIME,
    "TEST_model_forwards":0,
})
print("[FIVE-SEED FACETGATE DEVELOPMENT GATE PASS]",{
    "scientific_regime":SCIENTIFIC_REGIME,
    "mean_target_delta_pp":mean_delta,
    "positive_seed_count":positive_count,
    "TEST_forwards":0,
})


In [ ]:
# CELL 7 — PIN EXACT INTERNVL3.5 MODEL REVISION ONCE
# TEST remains unopened.
import os
from pathlib import Path
from huggingface_hub import snapshot_download

token=None
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token=os.environ.get("HF_TOKEN")

MODEL_DIR=Path("/kaggle/temp")/f"internvl35_facetgate_final_{MODEL_REVISION[:12]}"
MODEL_DIR.mkdir(parents=True,exist_ok=True)

if not any(MODEL_DIR.iterdir()):
    print("[MODEL SNAPSHOT] downloading pinned revision",MODEL_ID,MODEL_REVISION,flush=True)
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
        token=token,
        local_dir_use_symlinks=False,
    )

weight_ok=(MODEL_DIR/"model.safetensors.index.json").is_file() or (MODEL_DIR/"model.safetensors").is_file()
if not weight_ok:
    raise RuntimeError("Pinned InternVL3.5 snapshot lacks model weights")
if not (MODEL_DIR/"config.json").is_file():
    raise RuntimeError("Pinned InternVL3.5 snapshot lacks config.json")

print("[PINNED INTERNVL3.5 SNAPSHOT PASS]",{
    "path":str(MODEL_DIR),"model_id":MODEL_ID,"revision":MODEL_REVISION,"TEST_forwards":0,
})


In [ ]:
# CELL 8 — EMBEDDED EXACT INTERNVL3.5 LOCKED-TEST FEATURE WORKER
# Mechanically derived from the frozen InternVL FacetGate TRAIN+validation feature worker.
# Architecture/serialization/numerical-repair semantics are unchanged; only split governance
# changes to the explicitly authorized 932-row TEST.

INTERNVL35_LOCKED_TEST_FEATURE_WORKER_SHA256 = 'b7db55d945e635cc179dfb9e4e076bd006c01da60a00461a1940e24f262dec28'
INTERNVL35_LOCKED_TEST_FEATURE_WORKER_ZLIB_B64 = 'eNrFPO2S2zhy//UUNK9yJtcczuftJXJ4dbMe2efd8XhqPPZVrFJxKREa0aJImiDlmVGUykPkCfMk6W4AJEB92N7NJa67HZJodDcajf4CoN60zBdWGE7rqi5ZGFrJosjLyoqyLK+iKskz3pOfcu5Zn3ieeVYZZXG+8KwiqmZpMvasWcTFw93EsyZ5VrH7Ct5Vz6xeFA9WxK2swE5ZDI/wvyL2rCovJzMFRy9+lhFo1v3qT+tsghxFKQK87BHr168vFc+vF9Ed88Sfl0nKes2Tf/n2/CK8vXl/9eL8dnARvn5z/mrwLrgta9brvXj5KsBx+WkexdyRg/Kv4a+Tc59ly6TMs6H98vzF4Db8+9ubXwY34Yu3Vy9fv7JHrl+yKA5xwI7r9m7Or34JkqwyOuJHgHxuXb59cX4ZboVpmwgSqFxebADR1/Dd648DAOolUwH2JDjpw5QknFk3dVYlCzYoy7x0pvbLaMKqV1HFrCmLcH4tdk+SKtnnOikZt27P7k+sQ6tFHJzAJOaVtaJPa9vtXQw+vH4xCMQsxGyZTBigntRx1F+1XAPkczlT2ORzVoUSWCBwe0JroIXFDsh8aOOTPXqG3aFzVvhfgRDoF1FWR2m4HQiFAoprjfM8xVb/jlWOHaVp/iVM88mc4VTxyvZeRilnrrtVcPYlQVq3g3e3VlRXs7xMHmktWIuE8yS7A7FMy2jBgiIW819E5ecaSBFDRcngHUgBq8kUyYGacJB7uIzSmnHHzqfTZJLAKErgK8lidm+jIqHQ6M2Jy7wg9XR7yG9A1Ib0X58XaVL57LNj01Dc0W/HDdJKWeYgHvdJ8C+nJ1ZeWqh1jkYqCCQhn9cLUHIC3C64F1GWZ8kEFqgQ9gGJkLBYcZlMAYkxQ0S5SwVmy9k1M1d5JnDC4CwGdgbELGmJ6TIVHcjh1/DN24vBZfjy7c3fz28uYO2+v7oNjkTT1fnt6w+Dpu389nbw5vr2HTQL+1KBUvJpXi5YyZWhOa+r/LrMJ4yD5D3rp6Ti51n80wMM4EWeTZM7MELIW/bh8mVewqc4EXbrFctYSZrUQrzJY5YKWgWbVorGNTyLpkJRCgy6PnYJQdeAwyRTi2GBXUK0YfbIA8GAHkzBAPIwz9IHmnWvKmtehSVb5BULJwAfiMXQG0ecBV9jfINsz4J/v5O2R0g+1xHMtVhp0IiCDDZl66CZBlUOz8agNYR1nI3pLSQMYfVQsMDOpme2wIv/GpCaszDO63HKBHQHwyRfFDXwFhMSYXGmQLE6/tEV2IRVg7VdBCvb7vOqVBZu7RH81r49Ek7QzOqGGFH4HokxiqMC5kAJMuEhwUTAspyp5xZh8xmsecftTcBs8mC4sEBNrQWsX9kM/61B/I5rwZpLwG7xKsrAIC88Q/fckbIDhAjW9/EOfzK4L9ikgqVWZwmYO1OFwe+X4H2neZ3F1qpFhz4EmwJ6HR6NerMkjllG3g0bfDHVPvpQOe2+AAl58iislHh/Qn1ISEyyEooWsLA7eFY8ShTCDFkr8Ya8NcvLr/I5y4Bi6RdRHIOZB/qgoXaZ3M0qG9loYcNwkkachyE8ZWAqw/BJ0NBq1qjdJ5XZL8yNXtID7yO2FoYUxQEeLqpACRtwz04w7gEP+TmEabgDLfIOjtFuN9ITADReHhZC1yYzFOJufhtBdrELkfat1SYnT7uwTz2w4Mz9FvaJuzCJN5hvpr4D9z3cq05f511BapzzaAnUJ+hGN7r5opc2GoQDLELEjNsCDw6/xQOrk5wiLe89o7hOskzTGAsQTuZFDqKxMOhQRNWoOqSDVUvxSYkj+YN1O2PWHRr3ZKKhBRd8gHTB2E2jOq3gS2aN4fUB1A8gqySF5WGxJSwoCI/quxn8YYCuEAxqfHGwgxBodlmhkfoWeJkJw77gsKMJjALiJuiEMWtM6YXf2ylafyvOHrBsTSHSCXk9dmbRg5dBfJgyD4xfWQVHUkmgIUgTXjlgxx3g04MPLkahBGw2iW8Q3WNHtLEJ2liIC+6YQ1g9tHaI4AAfJPizY0kK/6EJix6GST95poGMgkA89cEP1GVmJWLqxcvBMY2FZegnHQh4JL4/WAOSFYdJi1IVmU7yIgGZiZgF5MnrCYppWqf47REmqpneVroKYQY4lgymLYJ0iy2KFHOGZxhEpRBLAimY7ZQdkKBbI2jNv0TlHfcJyZcELAElXH5ewBCBX2VnhCNzMWlLFppMsDFIFmj/l6yEOP3m1U8Q/8FIHsCrERysPh5ssdBOgwX9L9GapCwC3QvzSQnEWs8PpjzkYDEwJBYWT485iEgJiSVJUQQDTcMiupd2q7U++A1ISFQNJRcGm0HkAJYCPtIngNrGOuRQNKE4NI/Pk6LLnQjNaWw1xPL3ASBCBYI/Pow2KcChMxiAZQ/fvrixMFz9t//+z/96/25gUV47sqkzpeEQ3xDXVcTnIQYAZU0pNORL9gshMAuRoO/tW/YzQVHIfgGMwwxBZLFqJGKXecrsvg1RVKlFV7YkZveHzTf8t7IxFAJ4mmtbmna7T3/W3nZY5AVA6U9fItZgR+JxLYR8r4k4Kor0IUQlDpUSt3qiRqNNu5yRzqSjvtw14S6am0VRdVVGLNEwTiZVp7tsAaaBJR7YRaXJqeE1FEsnWBkiEOLhstHurzacR5/46EiORKV10qMXkKcIX7Q+8lHo2L1f5IVjp9EYdEo5J61B+D+cGlLtTYDFItwOI20ZVRpiiJXb9eHZ4DRhVlG+C9BM+FAk95A7iBzWXgv9E7l2gNktBMgK0wFkss69P2cPENlKowyrQ0K35mV/GNAasWnC0pg3/a2VfFo/t5AGOE1BvyGJjrMnDRifB/fGyse8HG1F12upMEIynI9hAaEvlmseuxA6kWP7CSiw04VVfg6BgaohspHPZ1HBILxuJNKh8S3hH6b/XXKYDuwTqqmKjYClS1kmvI5SCDHyBaQ6DyoysY1eU4OjYNXhfG2pkC9YIf9Pt/P/dART1kXchD7dYWlIj7d002QbkmSDVVUXKQPBP9Ubn0rBg1Y0KKR6KD+NtvdAs73P0ODyIsqeAwuTtI6l40XTesgfOEy9sE3cgoxJuEkLrNmEzfI0ZqV0uBPQt62uUZpMb5fj2+KqQFsQnaDXJor4DTTKw2hIC2aIsmxrWTFCJvyi1ia0mWsEugG7BC/9XNAEbABAnb2jRo/5vx5ZtE67IBxCrL8ER3u1s1VIEWB254EEQ7OhLMBhtBgnd3Vec1uwUNEyF+k8yDfnYZrMURH0MXl60o+1La3vsO/xPqfAj+QykoVnGrqYoN+IHoPRCrIYCEe3RRoQxjcaAGhaQcmP1r8H1nbzVTWWCiZAlupwKNYfhdVz/Sh7aO3UN04B+ikh6xwCvjQq+BbttnfQlTz/Xsq4OiLQgsb9l400ZEinzZz1RxJQx1uNaAachk/M3AhcsPY9LBERiFWqB0lWxv73HonZI7SUBoBWZCwqqUilCjvjaDIfg6/1OMTMTKW/eV4FqgXr3yGWlkIqCYlS0CzilLQ23W0TypbxpWpvVA1rD54gThkQUPLxW9xwpFuJlAfi85b6hZ/mX8BUuXqCZOPYbJImIIfuZI5sOW69oW9Ya5zRJKtZ8zHOKzTrU9tfIbW1b+tkSFZBABFPdldTNGpg+xJl2Nn2VbuQiW8jbYFZ8OWDa0OVqHAkfsOeAbKA3DlBBLAAYYF+An+Tb8A25FmqsdeA7+JvF+5teFVRwVTMD+jBhFoKBdLFJOiY3R4wVNK1oCdUE/cWjMJpRzOF9ZK6QQVC4S2+QaeNRUZdvxLhXeVtorsiHGvr7AAoWIKYJQkJZHLVFWWSl8FwtEvTO4QJHNMNlsWOI8A8VQqSr7YhEBkT7wQSsTNHSUpQt50NuYrM0jT916wv8K+ids357CDWIIWH0gwSDVlOd8moqNOZcoJo5ljJWUy9N02jO+GlAMgoi1C7qn/hyPeyuGPsAn2Lg5bf/65AkIQcP6VbrU8QJetWc/+xdjivK1DU5xZnLAswFXLlrsQUM1dnMr0zy07wYUMyiYBTZghRGRKDRs/GEULGAnk0W8BIKRvuygt7+khWYnSfI39qZeCXnkypmzqv+oviwTlB2anVYWtbAGoZ/Dbng+5kqwXXnIIlkfrAK8fylWObPbqj1YfRKtUGs9JYgSC+MRJQVK1GpyypU63VM8MDwC11kfaXwqpkzMnHnyTDwIGIFHHviAoR1EhzDA/4mfphkYCqtEbQIkkAHAG1cY6WGUCrR/nQZkeRJumM3bs0gfcof+RjB75OliHRDfdgGu3AhGWZTUyreV/HtRS45t5SYqOQErRo3TPF0Ov9tT3S4svHRZTBjJViEhKaySX6YYgkuTQKdCakWdspW7JUMmXbIiJ5SYXZA1n0ZvdsUuPsH+B+qYWHCJISlCOGqNiDSBHSViy6U7EcBjsGec20zQDCSKXdw3d/ux1YuDlYi9pwAcnW5EEJ5Ejl5i+vj3+U345bJRTO8tXgzRsOEKcnEuLEohEA5DOL4dC3qK0EPZWgJwD6+XB+uLRkfIK8gOqCbTVQn0n4U4BvIhkCgA/0Z1wm8R07jOoqn0S8suKE43aoCHHA41P1yrN4hNYK1jVJDMQKq7vCx7yAdYexvgfiA/eFqYaH65/ODMRy0jANz+6YzK/VJBFvVDsRUyg+Rrh0ZNTwnLjWXsHwaW+zHIvYMrj4B3mFqnwwjB+x+he9ctPhel/M1obGnraJfXribqFw0onFp3c8+KqD7HZRwiLHoyzsbickjS0tX3JvGdEdmTUc0eiFsjnshB4aeX8blcAmLbe3jPm0v0Hpu5zTdzkptNnkpUo6JuPYvu0du8OD4xFCruzPFPzbnj1XD0vxsN4kSBtfqI3KQ8t4p2R3YHtZGcJQvkRlHCKQk0aLcRxZCy/xcsNy5npw2orlzCSorYm9ytYmOru07XcJeU8y+NV50HK5TqDAFuOmPGm7/zeS1tjtcLM1FbSpmutsZe0bskca9e78vSV+3RLcoOX+P+sgbUoKrVK+w5Hnd8QhITyraHuMDtfEsjy6yTPlvXvS6W8ks+1c0D5i0ySL0lQz7LgIZqT6MNqSs9ghSXaQzHw8X7VkWnVFGUMICfaYw52mELo1gDuzP7Xa3a+Btl7IlUEschtSFZJDiCeYwm9xsFo38b8M1O5VmGb4OyrBdULe5UZyBxiH81GwNCsuc6y26Hspu7r5VS6Pem1O5emJl4GrG+NZRNy0ajdxt6vMVqw7MMhQlLoIicnjUXKpOPdGdCkk1xFqr6OnWR7elVGsi5Fa98WyZBkNUuofJqV4lMv54Qei7m3sj+Kxu0k0mTWHDc1sBswvr0J16gvUDg09Smhn7bXXI5tknYvTck6W+W/IkkjWSLFgNEkVhg5n6dSbeePg5E8/enFw5B8faSPgdYG1SL8BdzG9TqcgpHIRAOLL6IGVV/DizNo5Vds7xpEV80wGDJpbv/oZq371SEV/9Wl6f/Vb2kSHVUjmHftc41ijFEdzSeUqB7h2PXh9Nbh879DTRZkXIDInpjcJNvZmnca2LqDUhKRwjwflZJH5maLuNMMFVZGrcqLO8or5MNPLbQlXd/cIUyo6fg3R+qSE+FukHbYr8zPMrQL4/7ALIAI5WpvfRYf4JJ3bQkNrHOlp+tbUcWu+/j4DRSloM1jOtsn3AZEAfXjAE6rSG+LK2DRiWrb5LWZss8Q9nwd48GXequOXWQJh0Xzu04kkFRgI3woRCvaYz4e4B6V9bUNm4HM4b2yjXmZ4d3t+Owi62iC4xIGKLeUZi2KujqwuogJP28s4elLUtveF4SkEeRRYHr7vXQTHRydnveZcl8wRt6yFdgGSidA1X3zxLrRV4rY2Fc/PQzLyj0J+h0qVIoKg7XTywwX00MGmNRnSTXtigvEZXhvYaQouDD701U4GzUAlDxHzoLGLF6DZzuq+rwzmhYfGkHq2RRWayzEodPkQ4r41t0fPhlo+aINywVqzR2uDmuxBSrBJUR/B8X5iJtomJkfEgY7m1JxiYmoT6s8GFFk1Xo+FAi+icg7y0QvW9IWOT4u2dmnOkoqrZdxZyrQ8NhczZUA5D+Y+7ltLamaAjOeJc44b2IR/OB/CK+0SS+j+SI9V6BR0xRvrLYA8/KavWcyNKUEWw61KmISQ1ikse0+OzKOCiYoXBCKQTLAhnWYPvh5jFZnq89v3Xd6I3XNrRajXz5UQg5V8gE+iPhOsaPOHBOcO+2dHo7U0lnIH3quz5mwGME1WJmzttwPMeGJgejDRngHCvKdFsYNfySf2odKatUwiS/K67jeHj5rTQBrGYNU+r83CrOjf6yHLCNiz7K51oysDzaR0W73hRgcfLOomFjx3aHdsm4m60wiYO18IcRcF4UWzZiJrDR3gwRcfjAH+jUNh3AgZ9SMMmsUzEWkNgEl7IwR6N8IjTKKJQnyD3uKBOko46qNsn9lLffWaGxbcJ4PG52H7AVA13QmZbtpMhHqLNzQACY/Rk3AZ9sxEZjR5QxOU0JmdxfS3dq8z9W0DTnv7JqZc6wZ41r3m3shwQx27SqTpgT6TckoaGRuyMcemUR9JA6QusijTWpAR9psCLlpWq/DVrUXKXEJHrv1emX/hAV0kSyDoGOI9wH6fri+O5FniXdffroJUnFTmEPHLYORDkBXiHI7jXJHDLZqK1HPrdm/rz+c3N3sAeh9/MlrPOt0/XhrNp91mk/ifTdwwiiS+D3Aww213AEc+ZFiYfkMnyPF+PEPfGNLNXAelpG4ELu7wwJZAI7QBr1aozqBkbnt2iyqSCphedkH22D084H5TVjwGxhVbigLktT2w73h4+3BqT/Hqaig/87CMsvmK7pn6gMBu0eHF3d+HDzHQ5R6dRZ9e8OoUqKNBrGnZeU2jPRV9DZE4zMKh8jHNXcUD464iLp7YilK8SfogqHFfO5JoX+SUGXCI8LNKbA1BV3nhUSR3/Dlmm7Q5hMlCmacpK59yawkJ6RQvBmANJmWVZEEFOzC6yJrijgjeAf4lvHn79lZWSEGlGK8XLFTDl/crqjzOxfENcQPiysU7VLg/MrVp8VlCsCO87vzL4EKw+e5v5zcX1vXl+ZVF63V1BfEA7UfL+7XAmLyz6U3Tms+kPvbENlhIVQc8jAtZT+L2j7r3MP7krgFUFprikGgMR2TZYoyIPiEoA3XHc92YvsS5d9zs1H8hHRb2A0fyyW1yxOZ+gdRv94m8qLp/c9e4qAoTq02+EIgqgKsJlHGEcfAr0O5+yOOC4lZHcKU2fqhukuEGSwYxaPsdBy521dCsH3nH3ol36p2NWq733X61ngXWcRv9elFVBTtqTtp1jkAeYYaYW90Ud/8IXV2fQzbDHplz1IJXAF59O7jK1hfijB3t3dOxu+6Zu10TQofsrPZGPJW+E/hEZaWW0GwZzNS5Vvy3DEV2H8yW4B1Hak/cX7AoAwb1BKMZmtal2t+lt6WE3JTmvE5J+Ruq1cug68EdNQDFhpmEVEHHwzvVXnj0/7LuyZO7RZ7ooaGsCkDi7wyXXjXyYlCKg2O3QWXi+jnQQghC88PymXPsHx3gs/tDZYKDTRBhhvPzdt7AVICFwA2KvgpEhvg2gojRbXaQcEGo5rXR/3GsRlbB4nSGehAjESEJ8dhq6cGxiXxLWttIwqSXBkZoJLC32faoAw5z1cZOEljl45oqSVOA1+ipVqoKSuK781nesFcnaIn3z2QmZh7Mmvez9zj2HlPvsRoZ60/07xwAkvZI7I13upvsj8EKznvtVrRhtzr3YAib3RdIzSZ537e/bWizztA6XcVdhO1dl/u7istI2zpW+zt+wuM123v+vL/nYyh0aHvnx/HXerfnW7f2T7/WX2rW9t67R90uKbwTTq7Dmy09ZYU8ZV6kruBKh+zBQ21uFEe7otX8oAjZb7GFoC4HUl2C9K+pTXzzRRXtKhD5hCYYE/56QZfywcVkB2LIeG6E7rlOjTM76pAOedqzXfdakjhYYQzxVL0+Ha2tzRg9WGHsQYCbjU9H7noDPa6iYGWupc2LKNvWZiAlRwAfhp9wIypmYPdmMKmToob/itzA1fIGY2f+FjtV39kJMyTs9/N39vv4E/Z6HH97N8z58GKOcyYxXBKG9DdgOJUYaMSP1W/A8GdVJNJi2SFGliKGGj0LjiWAEcGqLXKlFlvzOukEd/3OiQjlesY2+ZFxlBmQd9Rq+PrqdnBzBWvj6v2bwc3rF+eX1s3g+vz1zQjD7XkgInxrj3Z3NfW3aLs4ebWiP2vbNFBtjqBpuzpSDJbgn07+FAR0bwjfAkr0Meh3u0PvJi3CGqD01ZUasAMrRLI+XDVY1hb8raJU5m4qpXE7yYsyg/ed1bfJaf8bbB502DXLT4KrHXVO/beMgM0DGb9DPghwzc8f7MK7tp4EFgwMrzz+AWSTpNYkzVEkYwaYmFVAFCXSwz5YS91g4u8RMK7OENANLnEPXeydpfkdXiyvyyU4A+73mgM+EZ1r6hke+4O6DnzrKY+KlkSohOYpP/7kGZ7v46WnebKPt15vbZbqZVqBNRHl2yLp2toZGUcxnf9z/mMLXATZcXDsqh9H2n854Trn1YH0GY3/ESoEMkxTPPyny1AcWepL/QI+aCJwT/H3l1BgLPhzEI90OqMkPXfUlqW3ZbmKApPXrHnoH/GoLKMHRxSNXE/8kJPWIApEMjoQkxl8oJpicOvRRAY0j2oGA5jAdv4CmD41e8HH257bW4DllbGizWFVLCK7b785fTudgpkMm+QuXB6HR6EoP4Sk2ko4JGtpSmyUiN1HkXj2l7xMY1uUDT0b5W33rySc+F2jBJq1nznCqpgnn0sm7gIZAM1HrE7jT5T1tV8rk5hFFBG2P5cRwsTfsZJMk+ywH0ZhMlcbKMHC7mM9s2muYX6bUo6UQ1+71W5rP48WtkUZWHLN9XbbdE8hmRCM0jOn47jczQ5djYJsH0Tc6Se7UZmGNpt1jynoUSfNje7qIs48g3o068yM3LDkLA9XC0fDraPD48OTw9ND9btRVOcR5wFV2QOo7zKVeh8R4jZH2fAMC5hy1XlH0QUr8c7vL2W6/pcywcsreGSHeKKyZQxRCndwAXm4nmHlnbge1ZewtGfX1fTgn+2eKuXZQ/q1Q+vl4Pz2/c1Alu9evH1zfTm4HYxsr4PTNfze/wAajPKb'

import base64, zlib, hashlib
WORKER_SOURCE=zlib.decompress(base64.b64decode(INTERNVL35_LOCKED_TEST_FEATURE_WORKER_ZLIB_B64)).decode("utf-8")
if hashlib.sha256(WORKER_SOURCE.encode("utf-8")).hexdigest()!=INTERNVL35_LOCKED_TEST_FEATURE_WORKER_SHA256:
    raise RuntimeError("Embedded InternVL3.5 locked-TEST worker SHA mismatch")
compile(WORKER_SOURCE,"internvl35_facetgate_locked_test_feature_worker.py","exec")
print("[INTERNVL3.5 LOCKED-TEST WORKER FREEZE PASS]",{
    "sha256":INTERNVL35_LOCKED_TEST_FEATURE_WORKER_SHA256,
    "test_rows":932,
    "world_size":2,
    "protocol_sha256":FACETGATE_COMMON_SHA256,
    "TEST_forwards":0,
})


In [ ]:
# CELL 9 — FINAL LOCKED TEST: FULL FACETGATE, FIVE SEEDS
# This is the ONLY cell that performs InternVL3.5 TEST model forwards.
#
# Full FacetGate:
#   Gated -> Offensive/Vulgar/Abuse/Sarcastic/Language
#   SHTE-derived Target specialist -> Target
#
# No training. No TEST threshold tuning. No TEST checkpoint selection.

import os, sys, subprocess, time, json, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    average_precision_score, hamming_loss
)

if not AUTHORIZE_LOCKED_TEST or LOCKED_TEST_ACK!="I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST":
    raise RuntimeError("Final TEST authorization missing.")
if not all(FIVE_SEED_GATE.values()):
    raise RuntimeError("Five-seed development gate is not frozen PASS.")

if not torch.cuda.is_available() or torch.cuda.device_count()!=2:
    raise RuntimeError(
        f"Final InternVL3.5 FacetGate TEST requires Kaggle T4x2. "
        f"cuda={torch.cuda.is_available()} count={torch.cuda.device_count()}"
    )

WORKER_PATH=WORK_ROOT/"internvl35_facetgate_locked_test_feature_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE,encoding="utf-8")
if sha256_file(WORKER_PATH)!=INTERNVL35_LOCKED_TEST_FEATURE_WORKER_SHA256:
    raise RuntimeError("Materialized InternVL3.5 final TEST worker SHA mismatch")

test_master=master[master.split.eq("test")].sort_values("official_row_index").reset_index(drop=True)
expected_test_idx=test_master.official_row_index.to_numpy(dtype=np.int64)
test_gold_bin=np.asarray(
    [[1 if str(r[t])=="Yes" else 0 for t in BINARY_TASKS] for r in test_master.to_dict("records")],
    dtype=np.int64
)
test_gold_lang=np.asarray([LANGUAGE_LABELS.index(str(x)) for x in test_master.language],dtype=np.int64)
test_gold_target=np.asarray([
    [1 if lab in set(json.loads(s)) else 0 for lab in TARGET_LABELS]
    for s in test_master.target_json
],dtype=np.int64)

def load_test_shards(seed, feature_dir, expected_fp):
    parts=[]; total_forwards=0
    for rank in [0,1]:
        npz=feature_dir/f"facet_features_rank{rank}.npz"
        js=feature_dir/f"facet_features_rank{rank}.json"
        if not npz.is_file() or not js.is_file():
            raise RuntimeError(f"Missing locked-TEST shard seed={seed} rank={rank}")
        meta=load_json(js)
        if meta.get("schema")!="M3Offend_FacetGate_v1_0_LOCKED_TEST_feature_shard":
            raise RuntimeError(f"Wrong locked-TEST shard schema seed={seed} rank={rank}")
        if int(meta.get("rank",-1))!=rank or int(meta.get("world",-1))!=2:
            raise RuntimeError(f"Rank/world drift seed={seed} rank={rank}")
        if int(meta.get("seed",-1))!=seed:
            raise RuntimeError(f"Seed drift in locked-TEST shard seed={seed}")
        if meta.get("model_id")!=MODEL_ID or meta.get("model_revision")!=MODEL_REVISION:
            raise RuntimeError(f"Model identity drift seed={seed}")
        if meta.get("parent_checkpoint_fingerprint")!=expected_fp:
            raise RuntimeError(f"Gated branch fingerprint drift seed={seed}")
        if int(meta.get("representation_dim",-1))!=1024 or not bool(meta.get("locked_test_authorized",False)):
            raise RuntimeError(f"Locked-TEST shard contract drift seed={seed}")
        total_forwards += int(meta.get("test_model_forwards",-1))
        with np.load(npz,allow_pickle=False) as z:
            parts.append({k:z[k] for k in z.files})

    merged={}
    for key in ["official_row_index","image_id","split","visual","text","joint","z_binary","z_language","z_target"]:
        merged[key]=np.concatenate([p[key] for p in parts],axis=0)
    order=np.argsort(merged["official_row_index"].astype(np.int64),kind="stable")
    for key in merged:
        merged[key]=merged[key][order]

    if total_forwards!=932:
        raise RuntimeError(f"Seed {seed} TEST model-forward count={total_forwards}, expected 932")
    if len(merged["official_row_index"])!=932 or len(np.unique(merged["official_row_index"]))!=932:
        raise RuntimeError(f"Seed {seed} TEST row duplication/loss")
    if not np.array_equal(merged["official_row_index"].astype(np.int64),expected_test_idx):
        raise RuntimeError(f"Seed {seed} ordered TEST identity mismatch")
    if not np.all(merged["split"].astype(str)=="test"):
        raise RuntimeError(f"Seed {seed} non-TEST row reached final ledger")
    return merged,total_forwards

def core5_metrics(zb,zl,gthr):
    bp=1.0/(1.0+np.exp(-np.clip(zb.astype(np.float64),-60,60)))
    acc=[]; mf=[]; tasks={}; preds={}
    for k,t in enumerate(BINARY_TASKS):
        th=float(gthr["binary"][t])
        pred=(bp[:,k]>=th).astype(np.int64)
        a=float(accuracy_score(test_gold_bin[:,k],pred))
        f=float(f1_score(test_gold_bin[:,k],pred,average="macro",zero_division=0))
        acc.append(a); mf.append(f); preds[t]=pred
        tasks[t]={"accuracy":a,"macro_f1":f,"threshold":th}
    lp=np.exp(zl.astype(np.float64)-zl.max(axis=1,keepdims=True))
    lp/=lp.sum(axis=1,keepdims=True)
    lpred=lp.argmax(axis=1)
    la=float(accuracy_score(test_gold_lang,lpred))
    lf=float(f1_score(test_gold_lang,lpred,average="macro",zero_division=0))
    acc.append(la); mf.append(lf)
    tasks["language"]={"accuracy":la,"macro_f1":lf}
    return {
        "accuracy":float(np.mean(acc)),
        "macro_f1":float(np.mean(mf)),
        "tasks":tasks,
    },bp,lp,preds,lpred

PER_SEED_RESULTS=[]
FRESH_TEST_FORWARD_COUNT=0

for seed in PUBLICATION_SEEDS:
    print("\n"+"="*100)
    print(f"FINAL LOCKED TEST — FULL FACETGATE — INTERNVL3.5 — SEED {seed}")
    print("="*100)

    bundle=BUNDLES[seed]
    seed_dir=WORK_ROOT/f"seed_{seed}"
    feature_dir=seed_dir/"locked_test_features"
    seed_dir.mkdir(parents=True,exist_ok=True)
    feature_dir.mkdir(parents=True,exist_ok=True)

    gcp=Path(GATED[seed]["path"])
    gated_fp=GATED[seed]["inference_fingerprint"]
    gthr=normalize_gated_thresholds(bundle["gated_threshold_path"])
    sthr=normalize_specialist_thresholds(bundle["specialist_threshold_path"])
    fthr=np.asarray([sthr[x] for x in TARGET_LABELS],dtype=np.float64)

    lock_payload={
        "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
        "scientific_regime":SCIENTIFIC_REGIME,
        "seed":seed,
        "model_id":MODEL_ID,
        "model_revision":MODEL_REVISION,
        "gated_step":bundle["gated_step"],
        "gated_inference_fingerprint":gated_fp,
        "shte_specialist_step":bundle["specialist_step"],
        "shte_specialist_checkpoint_sha256":bundle["specialist_checkpoint_sha256"],
        "shte_validation_threshold_sha256":bundle["specialist_threshold_sha256"],
        "facetgate_protocol_sha256":FACETGATE_COMMON_SHA256,
    }
    lock_id=json_sha(lock_payload)
    marker=seed_dir/"LOCKED_TEST_ONCE_MARKER.json"

    reuse=False
    if marker.is_file():
        old=load_json(marker)
        metrics_path=seed_dir/"locked_test_metrics.json"
        ledger_path=seed_dir/"locked_test_prediction_ledger.parquet"
        if (
            old.get("lock_id")==lock_id
            and int(old.get("test_rows",-1))==932
            and metrics_path.is_file() and ledger_path.is_file()
            and sha256_file(ledger_path)==old.get("ledger_sha256")
        ):
            reuse=True
            print("[REUSE IDENTICAL COMPLETED LOCK] No repeated TEST forward",lock_id)

    if not reuse:
        # Once-only governance: a partial/mismatched prior TEST attempt cannot be silently resumed.
        if feature_dir.exists() and any(feature_dir.iterdir()):
            raise RuntimeError(
                f"Seed {seed} has partial locked-TEST feature files without an identical verified completion marker. "
                "TEST will not be forwarded again automatically. Use a fresh WORK_ROOT only after an explicit audit."
            )
        cfg={
            "seed":seed,
            "model_id":MODEL_ID,
            "model_revision":MODEL_REVISION,
            "model_path":str(MODEL_DIR),
            "adapter_path":str(gcp/"adapter"),
            "heads_path":str(gcp/"classifier_heads.pt"),
            "prepared_manifest":str(PREPARED_MANIFEST),
            "feature_dir":str(feature_dir),
            "allow_locked_test":True,
            "task_instruction":TASK_INSTRUCTION,
            "expected_hidden":EXPECTED_HIDDEN,
            "binary_tasks":BINARY_TASKS,
            "max_ocr_tokens":384,
            "image_tokens_per_patch":256,
            "expected_image_token_id":151671,
            "parent_checkpoint_fingerprint":gated_fp,
        }
        cfgp=seed_dir/"locked_test_worker_config.json"
        cfgp.write_text(json.dumps(cfg,indent=2),encoding="utf-8")

        env=os.environ.copy()
        env["FACET_WORKER_CONFIG"]=str(cfgp)
        env["TOKENIZERS_PARALLELISM"]="false"
        cmd=[
            sys.executable,"-u","-m","torch.distributed.run",
            "--standalone","--nproc_per_node=2",str(WORKER_PATH)
        ]
        print("[LOCKED TEST WORKER]", " ".join(cmd), flush=True)
        t0=time.time()
        rc=subprocess.run(cmd,env=env,cwd=str(seed_dir)).returncode
        if rc!=0:
            raise RuntimeError(f"Seed {seed} final TEST feature worker failed rc={rc}")
        print(f"[LOCKED TEST WORKER COMPLETE] seed={seed} seconds={time.time()-t0:.1f}")

        M,seed_forwards=load_test_shards(seed,feature_dir,gated_fp)
        FRESH_TEST_FORWARD_COUNT += seed_forwards

        V=M["visual"].astype(np.float32)
        T=M["text"].astype(np.float32)
        J=M["joint"].astype(np.float32)
        ZB=M["z_binary"].astype(np.float32)
        ZL=M["z_language"].astype(np.float32)
        ZG=M["z_target"].astype(np.float32)

        # Internal Gated branch of the full FacetGate architecture.
        c5,bp,lp,bpred,lpred=core5_metrics(ZB,ZL,gthr)

        # Diagnostic Gated Target baseline.
        pg=1.0/(1.0+np.exp(-np.clip(ZG.astype(np.float64),-60,60)))
        gtarget_thr=np.asarray([gthr["target"][lab] for lab in TARGET_LABELS],dtype=np.float64)
        gated_target,gated_target_pred=target_metrics(test_gold_target,pg,gtarget_thr)

        # SHTE-derived Target specialist inside full FacetGate.
        state=torch.load(bundle["specialist_checkpoint_path"],map_location="cpu",weights_only=False)
        if int(state.get("step",-1))!=int(bundle["specialist_step"]):
            raise RuntimeError(f"Seed {seed} frozen SHTE specialist step drift")

        specialist=SHTETargetSpecialist().cpu().eval()
        specialist.load_state_dict(state["facetgate_state"],strict=True)

        probs=[]; selectors=[]; views=[]
        with torch.no_grad():
            for st in range(0,932,128):
                sl=slice(st,min(st+128,932))
                zs,ex=specialist(
                    torch.from_numpy(V[sl]),torch.from_numpy(T[sl]),
                    torch.from_numpy(J[sl]),torch.from_numpy(ZG[sl])
                )
                probs.append(torch.sigmoid(zs).numpy())
                selectors.append(ex["selector"].numpy())
                views.append(ex["view_weights"].numpy())

        pf=np.concatenate(probs,axis=0)
        sel=np.concatenate(selectors,axis=0)
        vw=np.concatenate(views,axis=0)
        facet_target,facet_target_pred=target_metrics(test_gold_target,pf,fthr)

        # Full FacetGate score = 5 Gated tasks + SHTE-derived Target.
        full6_acc=float((5*c5["accuracy"]+facet_target["exact_set_accuracy"])/6.0)
        full6_mf1=float((5*c5["macro_f1"]+facet_target["macro_f1"])/6.0)

        result={
            "schema":"M3Offend_InternVL35_FacetGate_v1_0_FINAL_LOCKED_TEST_SEED",
            "lock_id":lock_id,
            "seed":seed,
            "scientific_regime":SCIENTIFIC_REGIME,
            "model_id":MODEL_ID,
            "model_revision":MODEL_REVISION,
            "test_rows":932,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "test_model_forwards":932,
            "facetgate":{
                "definition":"Gated Core-5 branch + SHTE-derived Target specialist",
                "gated_core5_branch":{
                    "selected_gated_step":bundle["gated_step"],
                    "gated_inference_fingerprint":gated_fp,
                    "metrics":c5,
                },
                "shte_target_specialist":{
                    "selected_step":bundle["specialist_step"],
                    "checkpoint_sha256":bundle["specialist_checkpoint_sha256"],
                    "validation_threshold_sha256":bundle["specialist_threshold_sha256"],
                    "target_metrics":facet_target,
                    "selector_mean":float(sel.mean()),
                    "selector_sd":float(sel.std()),
                    "view_weight_mean":dict(zip(["visual","text","product","difference"],vw.mean(axis=0).tolist())),
                },
                "full6_average_accuracy":full6_acc,
                "full6_macro_f1":full6_mf1,
            },
            "diagnostic_gated_target_baseline":{
                "target_metrics":gated_target,
                "target_macro_f1_delta_pp_to_facetgate":float(100*(facet_target["macro_f1"]-gated_target["macro_f1"])),
                "target_map_delta_to_facetgate":float(facet_target["map"]-gated_target["map"]),
            },
            "no_test_training":True,
            "no_test_threshold_calibration":True,
            "no_test_checkpoint_selection":True,
        }

        ledger=pd.DataFrame({
            "official_row_index":expected_test_idx,
            "image_id":test_master.image_id.astype(str).to_numpy(),
            "seed":seed,
        })
        for k,t in enumerate(BINARY_TASKS):
            ledger[f"gold_{t}"]=test_gold_bin[:,k]
            ledger[f"facetgate_gated_prob_{t}"]=bp[:,k]
            ledger[f"facetgate_gated_pred_{t}"]=bpred[t]
            ledger[f"facetgate_gated_threshold_{t}"]=float(gthr["binary"][t])

        ledger["gold_language"]=[LANGUAGE_LABELS[i] for i in test_gold_lang]
        ledger["facetgate_gated_pred_language"]=[LANGUAGE_LABELS[i] for i in lpred]
        for k,lab in enumerate(LANGUAGE_LABELS):
            ledger[f"facetgate_gated_prob_language_{lab}"]=lp[:,k]

        for j,lab in enumerate(TARGET_LABELS):
            ledger[f"gold_target_{lab}"]=test_gold_target[:,j]
            ledger[f"diagnostic_gated_target_prob_{lab}"]=pg[:,j]
            ledger[f"diagnostic_gated_target_pred_{lab}"]=gated_target_pred[:,j]
            ledger[f"facetgate_target_prob_{lab}"]=pf[:,j]
            ledger[f"facetgate_target_pred_{lab}"]=facet_target_pred[:,j]
            ledger[f"facetgate_target_threshold_{lab}"]=float(fthr[j])
            ledger[f"facetgate_selector_{lab}"]=sel[:,j]

        ledger["facetgate_target_prediction_json"]=[
            json.dumps([lab for j,lab in enumerate(TARGET_LABELS) if row[j]],ensure_ascii=False)
            for row in facet_target_pred
        ]

        ledger_path=seed_dir/"locked_test_prediction_ledger.parquet"
        ledger.to_parquet(ledger_path,index=False)
        ledger.to_csv(seed_dir/"locked_test_prediction_ledger.csv",index=False)
        atomic_json(seed_dir/"locked_test_metrics.json",result)
        atomic_json(marker,{
            "schema":"M3Offend_InternVL35_FacetGate_TEST_ONCE_MARKER_v1_0",
            "lock_id":lock_id,
            "seed":seed,
            "test_rows":932,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "test_model_forwards":932,
            "ledger_sha256":sha256_file(ledger_path),
            "metrics_sha256":sha256_file(seed_dir/"locked_test_metrics.json"),
        })

        del specialist,state,V,T,J,ZB,ZL,ZG,pf,pg,sel,vw,M
        gc.collect()

    result=load_json(seed_dir/"locked_test_metrics.json")
    PER_SEED_RESULTS.append(result)
    print("[FULL FACETGATE SEED COMPLETE]",{
        "seed":seed,
        "Core5_MF1":result["facetgate"]["gated_core5_branch"]["metrics"]["macro_f1"],
        "Target_MF1":result["facetgate"]["shte_target_specialist"]["target_metrics"]["macro_f1"],
        "Full6_MF1":result["facetgate"]["full6_macro_f1"],
        "test_rows":932,
    })

print("[ALL FIVE INTERNVL3.5 FACETGATE LOCKED TEST SEEDS COMPLETE]")


In [ ]:
# CELL 10 — FIVE-SEED FINAL FACETGATE AGGREGATION + PAPER-READY RESULT PACKAGE
# CPU only. No additional TEST model forward.
import pandas as pd, numpy as np, math, shutil
from scipy.stats import t as student_t

rows=[]
for r in PER_SEED_RESULTS:
    f=r["facetgate"]
    c5=f["gated_core5_branch"]["metrics"]
    tm=f["shte_target_specialist"]["target_metrics"]
    gb=r["diagnostic_gated_target_baseline"]["target_metrics"]
    rows.append({
        "seed":int(r["seed"]),
        "core5_accuracy":float(c5["accuracy"]),
        "core5_macro_f1":float(c5["macro_f1"]),
        "facetgate_target_macro_f1":float(tm["macro_f1"]),
        "facetgate_target_map":float(tm["map"]),
        "facetgate_target_exact_set_accuracy":float(tm["exact_set_accuracy"]),
        "facetgate_target_micro_f1":float(tm["micro_f1"]),
        "facetgate_target_samples_f1":float(tm["samples_f1"]),
        "facetgate_target_hamming_loss":float(tm["hamming_loss"]),
        "facetgate_rare_target_macro_f1":float(tm["rare_target_macro_f1"]),
        "diagnostic_gated_target_macro_f1":float(gb["macro_f1"]),
        "target_macro_f1_delta_pp":float(r["diagnostic_gated_target_baseline"]["target_macro_f1_delta_pp_to_facetgate"]),
        "facetgate_full6_average_accuracy":float(f["full6_average_accuracy"]),
        "facetgate_full6_macro_f1":float(f["full6_macro_f1"]),
    })

per_seed=pd.DataFrame(rows).sort_values("seed").reset_index(drop=True)
if per_seed.seed.tolist()!=sorted(PUBLICATION_SEEDS):
    raise RuntimeError("Five-seed final result identity drift")

def summarize_metric(values):
    x=np.asarray(values,dtype=np.float64)
    n=len(x); mean=float(x.mean()); sd=float(x.std(ddof=1))
    crit=float(student_t.ppf(0.975,n-1)); half=crit*sd/math.sqrt(n)
    return {
        "n":n,"mean":mean,"sample_sd":sd,
        "ci95_low":mean-half,"ci95_high":mean+half,
        "min":float(x.min()),"max":float(x.max()),
    }

summary=[]
for col in per_seed.columns:
    if col=="seed": continue
    summary.append({"metric":col,**summarize_metric(per_seed[col].to_numpy())})
summary_df=pd.DataFrame(summary)

final_dir=WORK_ROOT/"FINAL_5SEED_FACETGATE_RESULTS"
final_dir.mkdir(parents=True,exist_ok=True)
per_seed.to_csv(final_dir/"internvl35_facetgate_locked_test_per_seed.csv",index=False)
summary_df.to_csv(final_dir/"internvl35_facetgate_locked_test_mean_sd_ci95.csv",index=False)

for seed in PUBLICATION_SEEDS:
    src=WORK_ROOT/f"seed_{seed}"
    dst=final_dir/f"seed_{seed}"
    if dst.exists(): shutil.rmtree(dst)
    dst.mkdir(parents=True)
    for name in [
        "locked_test_metrics.json",
        "locked_test_prediction_ledger.parquet",
        "locked_test_prediction_ledger.csv",
        "LOCKED_TEST_ONCE_MARKER.json",
        "locked_test_worker_config.json",
    ]:
        p=src/name
        if p.is_file(): shutil.copy2(p,dst/name)


# -------------------------------------------------------------------------
# PAPER-READY REPORTING LAYER
# -------------------------------------------------------------------------
# IMPORTANT:
#   * Uses only the already-created locked-TEST prediction ledgers.
#   * Performs NO model forward.
#   * Performs NO checkpoint selection.
#   * Performs NO threshold calibration.
#   * Performs NO TEST-derived tuning.
#
# Outputs:
#   - binary No/Yes class-wise support / precision / recall / F1
#   - Language Bangla/English/Banglish class-wise support / precision / recall / F1
#   - Target label-wise support / precision / recall / F1 / AP / threshold
#   - per-task 5-task (Core-5) and 6-task FacetGate tables
#   - language-stratified content-task performance
#   - confusion-matrix long tables
#   - five-seed mean / sample-SD / 95% Student-t CI
# -------------------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    average_precision_score,
)

def _five_seed_ci(values):
    x=np.asarray(values,dtype=np.float64)
    n=len(x)
    if n != 5:
        raise RuntimeError(f"Paper-ready summary expected 5 seeds, got n={n}")
    mean=float(x.mean())
    sd=float(x.std(ddof=1))
    crit=float(student_t.ppf(0.975,n-1))
    half=float(crit*sd/math.sqrt(n))
    return {
        "n_seeds":n,
        "mean":mean,
        "sample_sd":sd,
        "ci95_low":mean-half,
        "ci95_high":mean+half,
        "min":float(x.min()),
        "max":float(x.max()),
    }

def _group_metric_summary(df, group_cols, metric_cols, support_col=None):
    rows=[]
    for key,g in df.groupby(group_cols, dropna=False, sort=True):
        if not isinstance(key,tuple):
            key=(key,)
        base=dict(zip(group_cols,key))
        if support_col is not None:
            supports=sorted({int(v) for v in g[support_col].tolist()})
            if len(supports)!=1:
                raise RuntimeError(
                    f"Locked TEST support changed across seeds for {base}: {supports}"
                )
            base["support"]=supports[0]
        for metric in metric_cols:
            stat=_five_seed_ci(g[metric].to_numpy(dtype=np.float64))
            rows.append({**base,"metric":metric,**stat})
    return pd.DataFrame(rows)

binary_class_rows=[]
language_class_rows=[]
target_class_rows=[]
confusion_rows=[]
task_rows=[]
core5_rows=[]
full6_rows=[]
language_stratified_rows=[]
seed_provenance_rows=[]

for seed in PUBLICATION_SEEDS:
    seed_dir=WORK_ROOT/f"seed_{seed}"
    ledger_path=seed_dir/"locked_test_prediction_ledger.parquet"
    if not ledger_path.is_file():
        raise RuntimeError(f"Missing paper ledger seed={seed}: {ledger_path}")
    led=pd.read_parquet(ledger_path).sort_values("official_row_index").reset_index(drop=True)

    if len(led)!=932 or led["official_row_index"].duplicated().any():
        raise RuntimeError(f"Paper ledger identity drift seed={seed}")

    # ------------------------------------------------------------------
    # Provenance table
    # ------------------------------------------------------------------
    b=BUNDLES[seed]
    seed_provenance_rows.append({
        "seed":seed,
        "scientific_regime":SCIENTIFIC_REGIME,
        "gated_step":int(b["gated_step"]),
        "gated_inference_fingerprint":GATED[seed]["inference_fingerprint"],
        "shte_specialist_step":int(b["specialist_step"]),
        "shte_specialist_checkpoint_sha256":b["specialist_checkpoint_sha256"],
        "shte_validation_threshold_sha256":b["specialist_threshold_sha256"],
        "test_rows":932,
        "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
    })

    # ------------------------------------------------------------------
    # Binary task class-wise detail + confusion matrices + task metrics
    # ------------------------------------------------------------------
    seed_task_acc=[]
    seed_task_mf1=[]

    for task in BINARY_TASKS:
        y=led[f"gold_{task}"].to_numpy(dtype=np.int64)
        p=led[f"facetgate_gated_pred_{task}"].to_numpy(dtype=np.int64)

        pr,rc,f1,sup=precision_recall_fscore_support(
            y,p,labels=[0,1],zero_division=0
        )
        for idx,cls in enumerate(["No","Yes"]):
            binary_class_rows.append({
                "seed":seed,
                "task":task,
                "class":cls,
                "support":int(sup[idx]),
                "precision":float(pr[idx]),
                "recall":float(rc[idx]),
                "f1":float(f1[idx]),
            })

        cm=confusion_matrix(y,p,labels=[0,1])
        for ti,tcls in enumerate(["No","Yes"]):
            for pi,pcls in enumerate(["No","Yes"]):
                confusion_rows.append({
                    "seed":seed,
                    "task":task,
                    "true_class":tcls,
                    "predicted_class":pcls,
                    "count":int(cm[ti,pi]),
                })

        acc=float(accuracy_score(y,p))
        mf=float(f1_score(y,p,labels=[0,1],average="macro",zero_division=0))
        task_rows.append({
            "seed":seed,
            "task":task.capitalize(),
            "task_type":"binary",
            "accuracy":acc,
            "macro_f1":mf,
            "map":np.nan,
        })
        seed_task_acc.append(acc)
        seed_task_mf1.append(mf)

    # ------------------------------------------------------------------
    # Language task: Bangla / English / Banglish individually
    # ------------------------------------------------------------------
    y_lang=led["gold_language"].astype(str).to_numpy()
    p_lang=led["facetgate_gated_pred_language"].astype(str).to_numpy()

    pr,rc,f1,sup=precision_recall_fscore_support(
        y_lang,p_lang,labels=LANGUAGE_LABELS,zero_division=0
    )
    for idx,lang in enumerate(LANGUAGE_LABELS):
        language_class_rows.append({
            "seed":seed,
            "class":lang,
            "support":int(sup[idx]),
            "precision":float(pr[idx]),
            "recall":float(rc[idx]),
            "f1":float(f1[idx]),
        })

    cm=confusion_matrix(y_lang,p_lang,labels=LANGUAGE_LABELS)
    for ti,tcls in enumerate(LANGUAGE_LABELS):
        for pi,pcls in enumerate(LANGUAGE_LABELS):
            confusion_rows.append({
                "seed":seed,
                "task":"language",
                "true_class":tcls,
                "predicted_class":pcls,
                "count":int(cm[ti,pi]),
            })

    lang_acc=float(accuracy_score(y_lang,p_lang))
    lang_mf=float(f1_score(
        y_lang,p_lang,labels=LANGUAGE_LABELS,average="macro",zero_division=0
    ))
    task_rows.append({
        "seed":seed,
        "task":"Language",
        "task_type":"multiclass",
        "accuracy":lang_acc,
        "macro_f1":lang_mf,
        "map":np.nan,
    })
    seed_task_acc.append(lang_acc)
    seed_task_mf1.append(lang_mf)

    # ------------------------------------------------------------------
    # Target: seven positive labels individually
    # ------------------------------------------------------------------
    y_target=np.column_stack([
        led[f"gold_target_{lab}"].to_numpy(dtype=np.int64)
        for lab in TARGET_LABELS
    ])
    p_target=np.column_stack([
        led[f"facetgate_target_pred_{lab}"].to_numpy(dtype=np.int64)
        for lab in TARGET_LABELS
    ])
    prob_target=np.column_stack([
        led[f"facetgate_target_prob_{lab}"].to_numpy(dtype=np.float64)
        for lab in TARGET_LABELS
    ])

    per_target_f1=[]
    per_target_ap=[]
    for j,lab in enumerate(TARGET_LABELS):
        yy=y_target[:,j]
        pp=p_target[:,j]
        prob=prob_target[:,j]
        precision,recall,f1_pos,support_pos=precision_recall_fscore_support(
            yy,pp,labels=[1],average=None,zero_division=0
        )
        ap=float(average_precision_score(yy,prob)) if len(np.unique(yy))>=2 else float("nan")
        threshold=float(led[f"facetgate_target_threshold_{lab}"].iloc[0])
        target_class_rows.append({
            "seed":seed,
            "target_label":lab,
            "support":int(support_pos[0]),
            "precision":float(precision[0]),
            "recall":float(recall[0]),
            "f1":float(f1_pos[0]),
            "average_precision":ap,
            "threshold":threshold,
        })
        per_target_f1.append(float(f1_pos[0]))
        per_target_ap.append(ap)

        cm=confusion_matrix(yy,pp,labels=[0,1])
        for ti,tcls in enumerate(["No","Yes"]):
            for pi,pcls in enumerate(["No","Yes"]):
                confusion_rows.append({
                    "seed":seed,
                    "task":f"Target::{lab}",
                    "true_class":tcls,
                    "predicted_class":pcls,
                    "count":int(cm[ti,pi]),
                })

    target_exact=float(np.mean(np.all(y_target==p_target,axis=1)))
    target_mf=float(np.mean(per_target_f1))
    target_map=float(np.nanmean(per_target_ap))
    task_rows.append({
        "seed":seed,
        "task":"Target",
        "task_type":"multilabel",
        "accuracy":target_exact,
        "macro_f1":target_mf,
        "map":target_map,
    })

    # Explicit Core-5 (4 binary + Language) and Full-6 (Core-5 + Target)
    core5_acc=float(np.mean(seed_task_acc))
    core5_mf=float(np.mean(seed_task_mf1))
    full6_acc=float(np.mean(seed_task_acc+[target_exact]))
    full6_mf=float(np.mean(seed_task_mf1+[target_mf]))

    core5_rows.append({
        "seed":seed,
        "offensive_accuracy":task_rows[-6]["accuracy"],
        "offensive_macro_f1":task_rows[-6]["macro_f1"],
        "vulgar_accuracy":task_rows[-5]["accuracy"],
        "vulgar_macro_f1":task_rows[-5]["macro_f1"],
        "abuse_accuracy":task_rows[-4]["accuracy"],
        "abuse_macro_f1":task_rows[-4]["macro_f1"],
        "sarcastic_accuracy":task_rows[-3]["accuracy"],
        "sarcastic_macro_f1":task_rows[-3]["macro_f1"],
        "language_accuracy":lang_acc,
        "language_macro_f1":lang_mf,
        "five_task_average_accuracy":core5_acc,
        "five_task_macro_f1":core5_mf,
    })

    full6_rows.append({
        "seed":seed,
        "offensive_accuracy":task_rows[-6]["accuracy"],
        "offensive_macro_f1":task_rows[-6]["macro_f1"],
        "vulgar_accuracy":task_rows[-5]["accuracy"],
        "vulgar_macro_f1":task_rows[-5]["macro_f1"],
        "abuse_accuracy":task_rows[-4]["accuracy"],
        "abuse_macro_f1":task_rows[-4]["macro_f1"],
        "sarcastic_accuracy":task_rows[-3]["accuracy"],
        "sarcastic_macro_f1":task_rows[-3]["macro_f1"],
        "language_accuracy":lang_acc,
        "language_macro_f1":lang_mf,
        "target_exact_set_accuracy":target_exact,
        "target_macro_f1":target_mf,
        "target_map":target_map,
        "six_task_average_accuracy":full6_acc,
        "six_task_macro_f1":full6_mf,
    })

    # Cross-check paper recomputation against final locked-test metrics JSON.
    metric_obj=load_json(seed_dir/"locked_test_metrics.json")
    archived_c5=metric_obj["facetgate"]["gated_core5_branch"]["metrics"]
    archived_target=metric_obj["facetgate"]["shte_target_specialist"]["target_metrics"]
    archived_full6=float(metric_obj["facetgate"]["full6_macro_f1"])
    if abs(core5_acc-float(archived_c5["accuracy"]))>1e-12:
        raise RuntimeError(f"Seed {seed} Core-5 accuracy reporting drift")
    if abs(core5_mf-float(archived_c5["macro_f1"]))>1e-12:
        raise RuntimeError(f"Seed {seed} Core-5 Macro-F1 reporting drift")
    if abs(target_mf-float(archived_target["macro_f1"]))>1e-12:
        raise RuntimeError(f"Seed {seed} Target Macro-F1 reporting drift")
    if abs(full6_mf-archived_full6)>1e-12:
        raise RuntimeError(f"Seed {seed} Full-6 Macro-F1 reporting drift")

    # ------------------------------------------------------------------
    # Performance stratified by GOLD meme language.
    #
    # To avoid an ill-defined single-class macro-F1 for the Language task,
    # the language-stratified table reports:
    #   - each binary task Accuracy/Macro-F1
    #   - Target exact-set Accuracy/Macro-F1
    #   - content-task (4 binary + Target) averages
    #   - Language classification accuracy on that stratum
    #   - all-six average Accuracy only
    # ------------------------------------------------------------------
    for lang in LANGUAGE_LABELS:
        mask=(y_lang==lang)
        n=int(mask.sum())
        if n<=0:
            raise RuntimeError(f"Seed {seed} has no TEST samples for language={lang}")

        b_acc=[]
        b_mf=[]
        row={"seed":seed,"gold_language":lang,"support":n}

        for task in BINARY_TASKS:
            yy=led.loc[mask,f"gold_{task}"].to_numpy(dtype=np.int64)
            pp=led.loc[mask,f"facetgate_gated_pred_{task}"].to_numpy(dtype=np.int64)
            a=float(accuracy_score(yy,pp))
            f=float(f1_score(yy,pp,labels=[0,1],average="macro",zero_division=0))
            row[f"{task}_accuracy"]=a
            row[f"{task}_macro_f1"]=f
            b_acc.append(a); b_mf.append(f)

        yt=y_target[mask]
        pt=p_target[mask]
        target_exact_lang=float(np.mean(np.all(yt==pt,axis=1)))
        target_f1_lang=[
            float(f1_score(yt[:,j],pt[:,j],zero_division=0))
            for j in range(7)
        ]
        target_mf_lang=float(np.mean(target_f1_lang))
        lang_cls_acc=float(accuracy_score(y_lang[mask],p_lang[mask]))

        row.update({
            "target_exact_set_accuracy":target_exact_lang,
            "target_macro_f1":target_mf_lang,
            "content5_average_accuracy":float(np.mean(b_acc+[target_exact_lang])),
            "content5_macro_f1":float(np.mean(b_mf+[target_mf_lang])),
            "language_classification_accuracy":lang_cls_acc,
            "full6_average_accuracy":float(np.mean(b_acc+[lang_cls_acc,target_exact_lang])),
        })
        language_stratified_rows.append(row)

# -------------------------------------------------------------------------
# Materialize paper tables.
# -------------------------------------------------------------------------
binary_class_df=pd.DataFrame(binary_class_rows).sort_values(["task","class","seed"]).reset_index(drop=True)
language_class_df=pd.DataFrame(language_class_rows).sort_values(["class","seed"]).reset_index(drop=True)
target_class_df=pd.DataFrame(target_class_rows).sort_values(["target_label","seed"]).reset_index(drop=True)
confusion_df=pd.DataFrame(confusion_rows).sort_values(["task","seed","true_class","predicted_class"]).reset_index(drop=True)
task_df=pd.DataFrame(task_rows).sort_values(["task","seed"]).reset_index(drop=True)
core5_df=pd.DataFrame(core5_rows).sort_values("seed").reset_index(drop=True)
full6_df=pd.DataFrame(full6_rows).sort_values("seed").reset_index(drop=True)
language_stratified_df=pd.DataFrame(language_stratified_rows).sort_values(["gold_language","seed"]).reset_index(drop=True)
seed_provenance_df=pd.DataFrame(seed_provenance_rows).sort_values("seed").reset_index(drop=True)

# Per-class five-seed summaries.
binary_class_summary=_group_metric_summary(
    binary_class_df,
    ["task","class"],
    ["precision","recall","f1"],
    support_col="support",
)
language_class_summary=_group_metric_summary(
    language_class_df,
    ["class"],
    ["precision","recall","f1"],
    support_col="support",
)
target_class_summary=_group_metric_summary(
    target_class_df,
    ["target_label"],
    ["precision","recall","f1","average_precision","threshold"],
    support_col="support",
)

# Per-task six-task summary.
task_summary=_group_metric_summary(
    task_df,
    ["task","task_type"],
    ["accuracy","macro_f1"],
    support_col=None,
)

# Core-5 and Full-6 summary tables.
def _wide_summary(df, id_col="seed"):
    records=[]
    for col in df.columns:
        if col==id_col:
            continue
        stat=_five_seed_ci(df[col].to_numpy(dtype=np.float64))
        records.append({"metric":col,**stat})
    return pd.DataFrame(records)

core5_summary=_wide_summary(core5_df)
full6_summary=_wide_summary(full6_df)

# Language-stratified five-seed summary.
lang_metric_cols=[
    c for c in language_stratified_df.columns
    if c not in {"seed","gold_language","support"}
]
language_stratified_summary=_group_metric_summary(
    language_stratified_df,
    ["gold_language"],
    lang_metric_cols,
    support_col="support",
)

# -------------------------------------------------------------------------
# Export all paper-ready CSVs.
# -------------------------------------------------------------------------
paper_files={
    "paper_seed_provenance.csv":seed_provenance_df,
    "paper_binary_classwise_per_seed.csv":binary_class_df,
    "paper_binary_classwise_5seed_summary.csv":binary_class_summary,
    "paper_language_classwise_per_seed.csv":language_class_df,
    "paper_language_classwise_5seed_summary.csv":language_class_summary,
    "paper_target_classwise_per_seed.csv":target_class_df,
    "paper_target_classwise_5seed_summary.csv":target_class_summary,
    "paper_confusion_matrices_long.csv":confusion_df,
    "paper_taskwise_per_seed.csv":task_df,
    "paper_taskwise_5seed_summary.csv":task_summary,
    "paper_core5_per_seed.csv":core5_df,
    "paper_core5_5seed_summary.csv":core5_summary,
    "paper_full6_per_seed.csv":full6_df,
    "paper_full6_5seed_summary.csv":full6_summary,
    "paper_language_stratified_per_seed.csv":language_stratified_df,
    "paper_language_stratified_5seed_summary.csv":language_stratified_summary,
}
for name,df in paper_files.items():
    df.to_csv(final_dir/name,index=False)

# A compact machine-readable index so manuscript/table generation can consume
# the final result package without guessing table semantics.
paper_index={
    "schema":"M3Offend_InternVL35_FacetGate_PAPER_READY_REPORT_v1_1",
    "model_name":"FacetGate",
    "model_definition":"Gated Core-5 branch + SHTE-derived Target specialist",
    "test_rows":932,
    "seeds":PUBLICATION_SEEDS,
    "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
    "statistics":"Five predefined seeds; mean, sample SD (ddof=1), two-sided 95% Student-t CI",
    "definitions":{
        "core5":"Offensive + Vulgar + Abuse + Sarcastic + Language",
        "full6":"Core-5 + multilabel Target",
        "target_accuracy":"exact-set accuracy",
        "target_macro_f1":"unweighted mean of positive-class F1 across the seven Target labels",
        "binary_classwise":"No and Yes reported separately with support, precision, recall, F1",
        "language_classwise":"Bangla, English, Banglish reported separately with support, precision, recall, F1",
        "language_stratified":"Content-task performance evaluated separately on gold Bangla, English, and Banglish meme subsets; Language-task macro-F1 is intentionally not computed within a single-gold-class stratum.",
    },
    "files":{name:{"rows":int(len(df)),"columns":list(df.columns)} for name,df in paper_files.items()},
    "no_additional_test_forward":True,
    "no_test_training":True,
    "no_test_threshold_calibration":True,
    "no_test_checkpoint_selection":True,
}
atomic_json(final_dir/"PAPER_READY_REPORT_INDEX.json",paper_index)

# Print the tables most likely to be pasted into the paper.
print("\n[PAPER READY] Core-5 five-seed summary")
display(core5_summary)
print("\n[PAPER READY] Full-6 five-seed summary")
display(full6_summary)
print("\n[PAPER READY] Binary No/Yes class-wise five-seed summary")
display(binary_class_summary)
print("\n[PAPER READY] Language Bangla/English/Banglish five-seed summary")
display(language_class_summary)
print("\n[PAPER READY] Target seven-label five-seed summary")
display(target_class_summary)
print("\n[PAPER READY] Gold-language-stratified five-seed summary")
display(language_stratified_summary)


summary_payload={
    "schema":"M3Offend_InternVL35_FacetGate_v1_0_5Seed_FINAL_LOCKED_TEST_SUMMARY",
    "model_name":"FacetGate",
    "model_definition":"Gated Core-5 branch + SHTE-derived Target specialist",
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "scientific_regime":SCIENTIFIC_REGIME,
    "facetgate_protocol_sha256":FACETGATE_COMMON_SHA256,
    "gated_protocol_sha256":PARENT_GATED_PROTOCOL_SHA256,
    "test_rows_per_seed":932,
    "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
    "seeds":PUBLICATION_SEEDS,
    "statistics":"mean + sample SD (ddof=1) + two-sided 95% Student-t CI across five predefined seeds",
    "primary_endpoint":"full FacetGate with frozen validation-selected internal components on the single locked TEST",
    "internal_components":{
        "core5":"Gated branch; frozen validation thresholds; no TEST tuning",
        "target":"SHTE-derived specialist; frozen validation checkpoint and seven validation thresholds",
    },
    "per_seed":rows,
    "summary":{r["metric"]:{k:r[k] for k in ["n","mean","sample_sd","ci95_low","ci95_high","min","max"]} for r in summary},
    "paper_ready_reporting":{
        "core5_per_seed":"paper_core5_per_seed.csv",
        "core5_5seed_summary":"paper_core5_5seed_summary.csv",
        "full6_per_seed":"paper_full6_per_seed.csv",
        "full6_5seed_summary":"paper_full6_5seed_summary.csv",
        "binary_classwise":"paper_binary_classwise_5seed_summary.csv",
        "language_classwise":"paper_language_classwise_5seed_summary.csv",
        "target_classwise":"paper_target_classwise_5seed_summary.csv",
        "language_stratified":"paper_language_stratified_5seed_summary.csv",
        "confusion_matrices":"paper_confusion_matrices_long.csv",
        "report_index":"PAPER_READY_REPORT_INDEX.json",
    },
    "no_test_training":True,
    "no_test_threshold_calibration":True,
    "no_test_checkpoint_selection":True,
    "no_selective_seed_rerun":True,
}
atomic_json(final_dir/"internvl35_facetgate_5seed_final_locked_test_summary.json",summary_payload)

manifest={
    "schema":"M3Offend_InternVL35_FacetGate_v1_0_FINAL_PACKAGE_MANIFEST",
    "pretest_asset_freeze_sha256":sha256_file(WORK_ROOT/"FINAL_PRETEST_ASSET_FREEZE.json"),
    "five_seed_development_gate_sha256":sha256_file(WORK_ROOT/"FIVE_SEED_DEVELOPMENT_GATE_PASS.json"),
    "files":{},
}
for p in sorted(final_dir.rglob("*")):
    if p.is_file() and p.name!="FINAL_PACKAGE_MANIFEST.json":
        manifest["files"][p.relative_to(final_dir).as_posix()]=sha256_file(p)
atomic_json(final_dir/"FINAL_PACKAGE_MANIFEST.json",manifest)

zip_base=WORK_ROOT/"M3Offend_9318_InternVL3_5_8B_HF448_FacetGate_5Seed_FINAL_LOCKED_TEST_v1_0"
zip_path=Path(shutil.make_archive(str(zip_base),"zip",root_dir=final_dir))

print("\nFULL FACETGATE — FIVE-SEED LOCKED TEST SUMMARY")
display(summary_df)
print("\nPER-SEED FULL FACETGATE LOCKED TEST")
display(per_seed)
print("\nFinal ZIP:",zip_path)
print("Final ZIP SHA256:",sha256_file(zip_path))
